# Additional Experiments

This notebook is the public, output-free entry point for reproducing the paper experiments.
Datasets and generated result files are intentionally excluded from the repository.


## Complementary boundary ablation


# Ablation 2: Complementary Boundary Q1 Analysis

`ablation2_complementary_boundary.py`를 E316 프로젝트 안에서 노트북으로 실행하기 위한 정리본입니다.

이 노트북은 다음을 수행합니다.
- `GH-ANFIS(base_only)` vs `GH-ANFIS(full)` 비교
- base confidence 기준 Q1 bin(low-confidence boundary samples) 분석
- fold별 bin 집계와 dataset/overall 요약 CSV 저장


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path):
    markers = [
        'model.py',
        'data.py',
        'learning.py',
        'utils.py',
        'ablation2_complementary_boundary.py',
    ]
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E316',
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_exp',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
from IPython.display import display

from ablation2_complementary_boundary import run_ablation

## Run Config

기본값은 E316에 복사해 둔 `no_mi` weight 기준입니다.
필요하면 아래 값만 수정한 뒤 실행하면 됩니다.

In [ ]:
RUN_CONFIG = {
    'mode': 'no_mi',
    'weight_root': 'hyper_parameter/cv_weights',
    'data_root': 'data',
    'n_folds': 5,
    'seed': 42,
    'n_bins': 4,
    'boundary_bin': 1,
    'out_dir': 'output/ablation2_complementary_boundary',
    'summary_check_path': None,  # 기존 summary와 비교하려면 경로 문자열로 바꾸세요.
    'project_root': PROJECT_ROOT,
    'verbose': True,
}

RUN_CONFIG

In [ ]:
results = run_ablation(**RUN_CONFIG)

fold_df = results['fold_df']
fold_metrics_df = results['fold_metrics_df']
q1_by_dataset_df = results['q1_by_dataset_df']
q1_overall_df = results['q1_overall_df']
failed_folds_df = pd.DataFrame(results['failed_folds'])
output_paths = {k: str(v) for k, v in results['output_paths'].items()}

output_paths

## Q1 Summary

In [ ]:
display(q1_by_dataset_df)
display(q1_overall_df)

## Fold-Level Detail

In [ ]:
display(fold_metrics_df)
display(fold_df.head(20))

## Repro Check / Failures

In [ ]:
results['repro_check']

In [ ]:
failed_folds_df if len(failed_folds_df) > 0 else 'No failed folds.'

## Feature-count summary


# 5-Fold Mean Feature Count by Dataset and Model

이 노트북은 `hyper_parameter/cv_weights`에 저장된 fold artifact를 읽어서, 각 데이터셋별로 각 모델이 실제로 사용한 feature 수를 5-fold 평균으로 집계합니다.

- 기본 집계 대상은 `mode == 'no_mi'` 입니다.
- `GH-ANFIS`는 artifact 내부 gate mask를 읽어 `base / residual / full` feature 수를 계산합니다.
- 나머지 모델은 artifact에 저장된 실제 입력 차원(`n_features`)을 사용합니다.
- `GA-ANFIS`, `PSO-ANFIS`는 설정된 `selected_features` 개수와 실제 사용된 입력 차원이 다를 수 있으므로 진단 표를 함께 출력합니다.

In [ ]:
from pathlib import Path
from typing import Optional
import warnings

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.base import InconsistentVersionWarning

warnings.filterwarnings('ignore', category=InconsistentVersionWarning)

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
WEIGHT_ROOT = ROOT / 'hyper_parameter' / 'cv_weights'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODE_FILTER = 'no_mi'
DATASET_ORDER = [
    'Breast_Cancer_Wisconsin_(Original)',
    'Vowel',
    'Spambase',
    'Gisette',
]
MODEL_ORDER = [
    'GH-ANFIS(base)',
    'GH-ANFIS(residual)',
    'GH-ANFIS(full)',
    'ANFIS',
    'GA-ANFIS',
    'PSO-ANFIS',
    'PH-ANFIS(Avg)',
    'PH-ANFIS(Stacked)',
    'SVM',
]

FILE_MODEL_MAP = {
    'gh_anfis': 'GH-ANFIS',
    'anfis': 'ANFIS',
    'ga_anfis': 'GA-ANFIS',
    'pso_anfis': 'PSO-ANFIS',
    'ph_anfis_avg': 'PH-ANFIS(Avg)',
    'ph_anfis_stacked': 'PH-ANFIS(Stacked)',
    'svm': 'SVM',
}

WEIGHT_ROOT

In [ ]:
def split_dataset_and_mode(dataset_value: str):
    dataset_value = str(dataset_value)
    if '__' not in dataset_value:
        return dataset_value, None
    dataset_name, mode = dataset_value.rsplit('__', 1)
    return dataset_name, mode


def load_artifact_payload(path: Path):
    if path.suffix == '.joblib':
        return joblib.load(path)
    return torch.load(path, map_location='cpu')


def _tensor_to_numpy(value):
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        return value
    if hasattr(value, 'detach'):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def gh_feature_counts_from_payload(payload: dict):
    state_dict = payload.get('state_dict') or {}
    params = payload.get('params') or {}

    def resolve_mask(prefix: str):
        hard_key = f'{prefix}_mask_hard'
        logits_key = f'{prefix}_mask_logits'
        threshold = params.get(f'{prefix}_mask_threshold')
        threshold = 0.5 if threshold is None else float(threshold)

        if hard_key in state_dict:
            mask = _tensor_to_numpy(state_dict[hard_key]).reshape(-1)
            return mask > 0
        if logits_key in state_dict:
            logits = state_dict[logits_key].detach().cpu()
            probs = torch.sigmoid(logits).numpy().reshape(-1)
            return probs >= threshold
        return np.ones(int(payload.get('n_features', 0)), dtype=bool)

    base_mask = resolve_mask('base')
    residual_mask = resolve_mask('residual')
    residual_mode = str(params.get('residual_gate_mode', 'complement')).strip().lower()
    if residual_mode == 'complement':
        residual_effective = residual_mask & (~base_mask)
    else:
        residual_effective = residual_mask

    full_mask = base_mask | residual_effective
    return {
        'GH-ANFIS(base)': int(base_mask.sum()),
        'GH-ANFIS(residual)': int(residual_effective.sum()),
        'GH-ANFIS(full)': int(full_mask.sum()),
    }


def collect_feature_counts(weight_root: Path, mode_filter: Optional[str] = 'no_mi'):
    rows = []
    for fold_dir in sorted(weight_root.glob('*/*')):
        if not fold_dir.is_dir() or not fold_dir.name.startswith('fold_'):
            continue

        for artifact_path in sorted(fold_dir.iterdir()):
            if artifact_path.suffix not in {'.pt', '.joblib'}:
                continue

            payload = load_artifact_payload(artifact_path)
            dataset_name, mode = split_dataset_and_mode(payload.get('dataset', fold_dir.parent.name))
            if mode_filter is not None and mode != mode_filter:
                continue

            file_stem = artifact_path.stem
            base_model_name = FILE_MODEL_MAP.get(file_stem, payload.get('model_name', file_stem))
            fold_value = int(payload.get('fold', str(fold_dir.name).split('_')[-1]))
            configured_selected = payload.get('params', {}).get('selected_features')
            configured_selected_len = len(configured_selected or []) if configured_selected is not None else np.nan

            if file_stem == 'gh_anfis':
                gh_counts = gh_feature_counts_from_payload(payload)
                for gh_model_name, feature_count in gh_counts.items():
                    rows.append({
                        'dataset': dataset_name,
                        'mode': mode,
                        'fold': fold_value,
                        'model': gh_model_name,
                        'feature_count': int(feature_count),
                        'count_basis': 'gh_gate_mask',
                        'configured_selected_len': np.nan,
                        'artifact_path': str(artifact_path.relative_to(ROOT)),
                    })
                continue

            rows.append({
                'dataset': dataset_name,
                'mode': mode,
                'fold': fold_value,
                'model': base_model_name,
                'feature_count': int(payload.get('n_features', 0)),
                'count_basis': 'artifact_n_features',
                'configured_selected_len': configured_selected_len,
                'artifact_path': str(artifact_path.relative_to(ROOT)),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df['dataset'] = pd.Categorical(df['dataset'], categories=DATASET_ORDER, ordered=True)
    df['model'] = pd.Categorical(df['model'], categories=MODEL_ORDER, ordered=True)
    return df.sort_values(['dataset', 'model', 'fold']).reset_index(drop=True)

In [ ]:
feature_counts_df = collect_feature_counts(WEIGHT_ROOT, mode_filter=MODE_FILTER)
feature_counts_df

In [ ]:
feature_count_summary_df = (
    feature_counts_df
    .groupby(['dataset', 'model'], observed=True, as_index=False)
    .agg(
        n_folds=('fold', 'nunique'),
        feature_count_mean=('feature_count', 'mean'),
        feature_count_std=('feature_count', 'std'),
        feature_count_min=('feature_count', 'min'),
        feature_count_max=('feature_count', 'max'),
        count_basis=('count_basis', lambda s: ', '.join(sorted(set(map(str, s))))),
    )
    .sort_values(['dataset', 'model'])
    .reset_index(drop=True)
)
feature_count_summary_df['feature_count_std'] = feature_count_summary_df['feature_count_std'].fillna(0.0)
feature_count_summary_df

In [ ]:
selection_diagnostics_df = (
    feature_counts_df
    .loc[feature_counts_df['model'].isin(['ANFIS', 'GA-ANFIS', 'PSO-ANFIS'])]
    .assign(configured_selected_len=lambda df: df['configured_selected_len'].fillna(0).astype(int))
    .assign(selection_gap=lambda df: df['feature_count'] - df['configured_selected_len'])
    .sort_values(['dataset', 'model', 'fold'])
    .reset_index(drop=True)
)
selection_diagnostics_df

In [ ]:
selection_gap_summary_df = (
    selection_diagnostics_df
    .groupby(['dataset', 'model'], observed=True, as_index=False)
    .agg(
        actual_feature_mean=('feature_count', 'mean'),
        configured_selected_mean=('configured_selected_len', 'mean'),
        selection_gap_mean=('selection_gap', 'mean'),
    )
    .sort_values(['dataset', 'model'])
    .reset_index(drop=True)
)
selection_gap_summary_df

In [ ]:
available_models = feature_count_summary_df['model'].dropna().astype(str).unique().tolist()
feature_count_pivot_mean = (
    feature_count_summary_df
    .pivot(index='dataset', columns='model', values='feature_count_mean')
    .reindex(index=DATASET_ORDER)
    .reindex(columns=[model for model in MODEL_ORDER if model in available_models])
)
feature_count_pivot_mean.round(1)

In [ ]:
feature_count_pivot_mean_std = (
    feature_count_summary_df
    .assign(mean_std=lambda df: df.apply(lambda row: f"{row['feature_count_mean']:.1f} ± {row['feature_count_std']:.1f}", axis=1))
    .pivot(index='dataset', columns='model', values='mean_std')
    .reindex(index=DATASET_ORDER)
    .reindex(columns=[model for model in MODEL_ORDER if model in available_models])
)
feature_count_pivot_mean_std

In [ ]:
summary_csv_path = OUTPUT_DIR / f'feature_count_summary_{MODE_FILTER or "all"}.csv'
pivot_csv_path = OUTPUT_DIR / f'feature_count_pivot_mean_{MODE_FILTER or "all"}.csv'
diagnostics_csv_path = OUTPUT_DIR / f'feature_count_diagnostics_{MODE_FILTER or "all"}.csv'

feature_count_summary_df.to_csv(summary_csv_path, index=False)
feature_count_pivot_mean.to_csv(pivot_csv_path)
selection_diagnostics_df.to_csv(diagnostics_csv_path, index=False)

print('saved:', summary_csv_path)
print('saved:', pivot_csv_path)
print('saved:', diagnostics_csv_path)

## Primary and complementary IF-THEN rules


# GH-ANFIS_E403 GRS Rule Viewer

- `Primary`는 GH-ANFIS의 `base` rule set입니다.
- `Complementary`는 GH-ANFIS의 `residual` rule set입니다.
- 각 규칙을 `IF ... THEN ...` 형태로 출력하고, 표(`DataFrame`)와 파일(`csv`, `txt`)로도 저장합니다.
- E403에서 저장된 `gh_anfis.pt` 체크포인트의 두 포맷을 모두 지원합니다.
  - `model_config` + `model_state_dict`
  - `n_features/n_outputs` + `params` + `state_dict`

`THEN` 절의 클래스는 학습 시 인코딩된 클래스 인덱스 기준입니다. 이진 분류에서는 `class_1 probability`를 함께 표시합니다.


In [ ]:
from pathlib import Path
import json
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()

def _looks_like_project_root(path: Path) -> bool:
    markers = ['model.py', 'data.py', 'learning.py', 'gh_config.py']
    return all((path / marker).exists() for marker in markers)

if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E403',
        PROJECT_ROOT / 'GH-ANFIS_E403',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS
from gh_config import normalize_gh_params
from data import (
    load_bcwd_data,
    load_gisette_data,
    load_spambase_data,
    load_vowel_data,
    coerce_numeric_frame,
    drop_nan_targets,
)

PROJECT_ROOT


In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CV_WEIGHT_ROOT = PROJECT_ROOT / 'hyper_parameter' / 'cv_weights'

AVAILABLE_GH_CHECKPOINTS = sorted(CV_WEIGHT_ROOT.glob('*/fold_*/gh_anfis.pt'))
AVAILABLE_DATASET_KEYS = sorted({path.parent.parent.name for path in AVAILABLE_GH_CHECKPOINTS})

DATASET_KEY = AVAILABLE_DATASET_KEYS[0] if AVAILABLE_DATASET_KEYS else 'Breast_Cancer_Wisconsin__Original___no_mi'
FOLD_IDX = 1
CHECKPOINT_PATH = None  # 예: PROJECT_ROOT / 'hyper_parameter' / 'cv_weights' / DATASET_KEY / 'fold_01' / 'gh_anfis.pt'

TOP_TERMS_PER_RULE = 8
COMPUTE_MEAN_RULE_ACTIVATION = True
EXPORT_DIR = PROJECT_ROOT / 'output' / 'grs_rule_if_then'

print('DEVICE =', DEVICE)
print('CV_WEIGHT_ROOT =', CV_WEIGHT_ROOT)
print('Available dataset keys:')
for key in AVAILABLE_DATASET_KEYS:
    print('  -', key)


In [ ]:
def sigmoid_np(x: np.ndarray | float) -> np.ndarray | float:
    x_clip = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x_clip))


def softmax_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    denom = np.sum(e)
    if denom <= 0:
        return np.full_like(e, 1.0 / max(len(e), 1), dtype=np.float64)
    return e / denom


def list_fold_checkpoints(dataset_key: str) -> list[Path]:
    return sorted((CV_WEIGHT_ROOT / str(dataset_key)).glob('fold_*/gh_anfis.pt'))


def resolve_checkpoint_path(dataset_key: str, fold_idx: int, checkpoint_path: str | Path | None = None) -> Path:
    if checkpoint_path is not None:
        path = Path(checkpoint_path).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f'Checkpoint not found: {path}')
        return path

    path = CV_WEIGHT_ROOT / str(dataset_key) / f'fold_{int(fold_idx):02d}' / 'gh_anfis.pt'
    if not path.exists():
        available = list_fold_checkpoints(dataset_key)
        raise FileNotFoundError(
            f'Checkpoint not found: {path}\nAvailable folds for {dataset_key}: {[p.parent.name for p in available]}'
        )
    return path.resolve()


def _infer_model_config(payload: dict[str, Any], params: dict[str, Any]) -> dict[str, Any]:
    cfg = payload.get('model_config')
    if cfg is not None:
        return dict(cfg)

    n_features = payload.get('n_features')
    n_outputs = payload.get('n_outputs')
    if n_features is None or n_outputs is None:
        raise KeyError('Checkpoint must contain either model_config or n_features/n_outputs.')

    return {
        'n_features': int(n_features),
        'n_outputs': int(n_outputs),
        'base_rules': int(params['base_rules']),
        'residual_rules': int(params['residual_rules']),
        'mf_per_feature': int(params['mf_per_feature']),
    }


def load_gh_checkpoint(checkpoint_path: Path, device: torch.device):
    payload = torch.load(checkpoint_path, map_location=device)
    if not isinstance(payload, dict):
        raise TypeError(f'Unsupported checkpoint payload type: {type(payload)}')

    raw_params = payload.get('hparams') or payload.get('params') or {}
    params = normalize_gh_params(raw_params)
    cfg = _infer_model_config(payload, params)

    state_dict = payload.get('model_state_dict') or payload.get('state_dict')
    if state_dict is None:
        raise KeyError('Checkpoint is missing model_state_dict/state_dict.')

    model = GH_ANFIS(
        n_features=int(cfg['n_features']),
        n_outputs=int(cfg['n_outputs']),
        base_rules=int(cfg['base_rules']),
        residual_rules=int(cfg['residual_rules']),
        mf_per_feature=int(cfg['mf_per_feature']),
        device=device,
        residual_gate_mode=str(params.get('residual_gate_mode', 'complement')),
        rule_init_mode=str(params.get('rule_init_mode', 'balanced')),
        rule_seed=int(params.get('rule_seed', 0)),
        firing_mode=str(params.get('firing_mode', 'htsk')),
        use_input_norm=bool(params.get('use_input_norm', False)),
        enable_residual_branch=bool(params.get('enable_residual_branch', True)),
    ).to(device)
    model.load_state_dict(state_dict)

    if params.get('base_mask_threshold') is not None:
        model.base_mask_threshold = float(params['base_mask_threshold'])
    if params.get('residual_mask_threshold') is not None:
        model.residual_mask_threshold = float(params['residual_mask_threshold'])

    model.base_mask_frozen = bool(params.get('base_hard_epochs', 0) or params.get('random_role_assignment', False))
    model.residual_mask_frozen = bool(params.get('residual_hard_epochs', 0) or params.get('random_role_assignment', False))

    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')

    feature_names = payload.get('feature_names')
    if feature_names is None:
        feature_names = [f'f{i}' for i in range(model.n_features)]
    else:
        feature_names = list(feature_names)

    return model, feature_names, params, payload


def load_snapshot_bundle(dataset_key: str):
    if str(dataset_key).startswith('Breast_Cancer_Wisconsin'):
        X_df, y, feature_names = load_bcwd_data()
    elif str(dataset_key).startswith('Vowel'):
        X_df, y, feature_names = load_vowel_data()
    elif str(dataset_key).startswith('Spambase'):
        X_df, y, feature_names = load_spambase_data()
    elif str(dataset_key).startswith('Gisette'):
        X_df, y, feature_names = load_gisette_data()
    else:
        raise ValueError(f'Unsupported dataset key: {dataset_key}')

    X_df = coerce_numeric_frame(X_df)
    X_df, y = drop_nan_targets(X_df, y)
    if not hasattr(X_df, 'columns'):
        X_df = pd.DataFrame(X_df, columns=feature_names)
    return X_df.copy(), np.asarray(y), list(X_df.columns)


def align_snapshot_to_checkpoint(X_df: pd.DataFrame, checkpoint_feature_names: list[Any]) -> pd.DataFrame:
    missing = [col for col in checkpoint_feature_names if col not in X_df.columns]
    if missing:
        raise KeyError(f'Missing columns in snapshot data: {missing[:10]}')
    return X_df.loc[:, checkpoint_feature_names].copy()


def apply_saved_scaler(X_df: pd.DataFrame, payload: dict[str, Any], device: torch.device) -> torch.Tensor:
    X_arr = np.asarray(X_df.values, dtype=np.float32)
    mean = payload.get('scaler_mean')
    scale = payload.get('scaler_scale')
    if mean is not None and scale is not None:
        mean_arr = np.asarray(mean, dtype=np.float32)
        scale_arr = np.asarray(scale, dtype=np.float32)
        safe_scale = np.where(scale_arr == 0, 1.0, scale_arr)
        X_arr = (X_arr - mean_arr) / safe_scale
    return torch.tensor(X_arr, dtype=torch.float32, device=device)


@torch.no_grad()
def collect_rule_activation_stats(model: GH_ANFIS, x_tensor: torch.Tensor, batch_size: int = 512):
    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')

    base_weights = []
    residual_weights = []
    n = int(x_tensor.shape[0])

    for start in range(0, n, batch_size):
        xb = x_tensor[start:start + batch_size]
        _, acts = model(xb, return_activations=True, use_soft_eval=False)
        bw = acts.get('base_rule_weights')
        rw = acts.get('residual_rule_weights')
        if bw is not None:
            base_weights.append(bw.detach().cpu().numpy())
        if rw is not None:
            residual_weights.append(rw.detach().cpu().numpy())

    base_mean = np.concatenate(base_weights, axis=0).mean(axis=0) if base_weights else None
    residual_mean = np.concatenate(residual_weights, axis=0).mean(axis=0) if residual_weights else None
    return base_mean, residual_mean


In [ ]:
def gh_branch_masks(model: GH_ANFIS) -> dict[str, np.ndarray]:
    base_soft = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
    base_hard = (base_soft >= float(model.base_mask_threshold)).astype(float)

    residual_soft = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
    residual_hard = (residual_soft >= float(model.residual_mask_threshold)).astype(float)

    if bool(model.residual_use_complement):
        residual_effective_hard = residual_hard * (1.0 - base_hard)
    else:
        residual_effective_hard = residual_hard.copy()

    return {
        'base_soft': base_soft,
        'base_hard': base_hard,
        'residual_soft': residual_soft,
        'residual_hard': residual_hard,
        'residual_effective_hard': residual_effective_hard,
    }


def _module_pack(model: GH_ANFIS, module: str) -> dict[str, Any]:
    masks = gh_branch_masks(model)
    if module == 'base':
        return {
            'module': 'base',
            'module_label': 'Primary',
            'display_prefix': 'P',
            'internal_prefix': 'B',
            'centers': model.s_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.s_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.base_consequents,
            'gate_hard': masks['base_hard'],
            'n_rules': int(model.base_rules),
        }
    if module == 'residual':
        return {
            'module': 'residual',
            'module_label': 'Complementary',
            'display_prefix': 'C',
            'internal_prefix': 'R',
            'centers': model.p_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.p_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.residual_consequents,
            'gate_hard': masks['residual_effective_hard'],
            'n_rules': int(model.residual_rules),
        }
    raise ValueError("module must be 'base' or 'residual'")


def infer_term_label(centers_1d: np.ndarray, mf_idx: int) -> tuple[str, int]:
    low_idx = int(np.argmin(centers_1d))
    high_idx = int(np.argmax(centers_1d))
    if mf_idx == low_idx:
        return 'low', -1
    if mf_idx == high_idx:
        return 'high', 1
    return f'mid(mf{int(mf_idx) + 1})', 0


def compute_rule_result(mp: dict[str, Any], rule_idx: int, x_proto: np.ndarray, n_outputs: int) -> tuple[np.ndarray, np.ndarray, int]:
    x_aug = np.concatenate([x_proto.astype(np.float32), np.array([1.0], dtype=np.float32)], axis=0)

    logits = []
    for out_idx in range(int(n_outputs)):
        layer = mp['consequents'][out_idx * mp['n_rules'] + rule_idx]
        weight = layer.weight.detach().cpu().numpy().reshape(-1)
        bias = 0.0
        if layer.bias is not None:
            bias = float(layer.bias.detach().cpu().numpy().reshape(-1)[0])
        logits.append(float(np.dot(weight, x_aug) + bias))

    logits = np.asarray(logits, dtype=np.float64)
    if int(n_outputs) == 1:
        p1 = float(sigmoid_np(logits[0]))
        probs = np.asarray([1.0 - p1, p1], dtype=np.float64)
        pred_class = int(p1 >= 0.5)
    else:
        probs = softmax_np(logits)
        pred_class = int(np.argmax(probs))
    return logits, probs, pred_class


def format_consequent_text(probs: np.ndarray, pred_class: int, logits: np.ndarray, n_outputs: int) -> str:
    if int(n_outputs) == 1:
        return (
            f'class_1 probability = {probs[1]:.4f}, '
            f'predicted_class = class_{pred_class}, '
            f'logit = {logits[0]:.4f}'
        )

    prob_text = ', '.join([f'class_{idx}={prob:.4f}' for idx, prob in enumerate(probs)])
    logit_text = ', '.join([f'class_{idx}={logit:.4f}' for idx, logit in enumerate(logits)])
    return f'predicted_class = class_{pred_class}, probs = [{prob_text}], logits = [{logit_text}]'


def module_rule_tables(
    model: GH_ANFIS,
    feature_names: list[Any],
    module: str,
    mean_rule_weights: np.ndarray | None = None,
    top_terms: int = 8,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    mp = _module_pack(model, module)

    selector_logits = np.asarray(mp['selector_logits'], dtype=np.float64)
    selector_logits = selector_logits - selector_logits.max(axis=-1, keepdims=True)
    selector_probs = np.exp(selector_logits)
    selector_probs = selector_probs / selector_probs.sum(axis=-1, keepdims=True)

    centers = np.asarray(mp['centers'], dtype=np.float64)
    gate_hard = np.asarray(mp['gate_hard'], dtype=np.float64)

    rule_rows = []
    term_rows = []

    for r in range(mp['n_rules']):
        best_mf = selector_probs[r].argmax(axis=1)
        best_prob = selector_probs[r].max(axis=1)

        x_proto = np.zeros(len(feature_names), dtype=np.float32)
        for d in range(len(feature_names)):
            x_proto[d] = float(centers[d, int(best_mf[d])] * gate_hard[d])

        logits, probs, pred_class = compute_rule_result(mp, r, x_proto, model.n_outputs)

        active_idx = np.where(gate_hard > 0.5)[0].tolist()
        if active_idx:
            active_idx = sorted(active_idx, key=lambda d: float(best_prob[d]), reverse=True)
        else:
            active_idx = np.argsort(best_prob)[::-1].tolist()
        active_idx = active_idx[: max(1, int(top_terms))]

        per_rule_terms = []
        for d in active_idx:
            term_label, term_sign = infer_term_label(centers[d], int(best_mf[d]))
            term_text = f'{feature_names[d]} is {term_label}'
            row = {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'feature': feature_names[d],
                'selector_prob': float(best_prob[d]),
                'selected_mf_idx': int(best_mf[d]),
                'selected_center': float(centers[d, int(best_mf[d])]),
                'term_label': term_label,
                'term_polarity_sign': int(term_sign),
                'term_text': term_text,
                'predicted_class': int(pred_class),
                'class_1_probability': float(probs[1]) if len(probs) > 1 else np.nan,
                'consequent_text': format_consequent_text(probs, pred_class, logits, model.n_outputs),
                'mean_rule_activation': float(mean_rule_weights[r]) if mean_rule_weights is not None and r < len(mean_rule_weights) else np.nan,
            }
            per_rule_terms.append(row)
            term_rows.append(row)

        antecedent_text = ' AND '.join([row['term_text'] for row in per_rule_terms]) if per_rule_terms else '(no active terms)'
        consequent_text = format_consequent_text(probs, pred_class, logits, model.n_outputs)
        if_then_text = f'IF {antecedent_text} THEN {consequent_text}'

        rule_rows.append(
            {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'active_feature_count': int(np.sum(gate_hard > 0.5)),
                'antecedent_text': antecedent_text,
                'consequent_text': consequent_text,
                'if_then_text': if_then_text,
                'predicted_class': int(pred_class),
                'class_1_probability': float(probs[1]) if len(probs) > 1 else np.nan,
                'class_probs': json.dumps({f'class_{idx}': float(prob) for idx, prob in enumerate(probs)}, ensure_ascii=False),
                'rule_logits': json.dumps({f'class_{idx}': float(logit) for idx, logit in enumerate(np.atleast_1d(logits))}, ensure_ascii=False),
                'mean_rule_activation': float(mean_rule_weights[r]) if mean_rule_weights is not None and r < len(mean_rule_weights) else np.nan,
            }
        )

    return pd.DataFrame(rule_rows), pd.DataFrame(term_rows)


def print_if_then_rules(rule_df: pd.DataFrame, title: str):
    print(f'[{title}]')
    if rule_df.empty:
        print('  (no rules)')
        return
    for _, row in rule_df.sort_values('rule_idx').iterrows():
        print(f"- {row['display_rule_id']} ({row['internal_rule_id']}): {row['if_then_text']}")


def export_rule_outputs(
    export_dir: Path,
    stem: str,
    primary_rules_df: pd.DataFrame,
    primary_terms_df: pd.DataFrame,
    complementary_rules_df: pd.DataFrame,
    complementary_terms_df: pd.DataFrame,
    checkpoint_info: dict[str, Any],
) -> dict[str, Path]:
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    primary_rules_csv = export_dir / f'{stem}_primary_rules.csv'
    primary_terms_csv = export_dir / f'{stem}_primary_terms.csv'
    complementary_rules_csv = export_dir / f'{stem}_complementary_rules.csv'
    complementary_terms_csv = export_dir / f'{stem}_complementary_terms.csv'
    summary_json = export_dir / f'{stem}_summary.json'
    if_then_txt = export_dir / f'{stem}_if_then_rules.txt'

    primary_rules_df.to_csv(primary_rules_csv, index=False)
    primary_terms_df.to_csv(primary_terms_csv, index=False)
    complementary_rules_df.to_csv(complementary_rules_csv, index=False)
    complementary_terms_df.to_csv(complementary_terms_csv, index=False)

    summary_payload = {
        'checkpoint_path': str(checkpoint_info['checkpoint_path']),
        'dataset_key': str(checkpoint_info['dataset_key']),
        'fold_idx': int(checkpoint_info['fold_idx']),
        'primary_rule_count': int(len(primary_rules_df)),
        'complementary_rule_count': int(len(complementary_rules_df)),
    }
    summary_json.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding='utf-8')

    lines = []
    lines.append('[Primary / base]')
    for _, row in primary_rules_df.sort_values('rule_idx').iterrows():
        lines.append(f"{row['display_rule_id']} ({row['internal_rule_id']}): {row['if_then_text']}")
    lines.append('')
    lines.append('[Complementary / residual]')
    for _, row in complementary_rules_df.sort_values('rule_idx').iterrows():
        lines.append(f"{row['display_rule_id']} ({row['internal_rule_id']}): {row['if_then_text']}")
    if_then_txt.write_text('\n'.join(lines), encoding='utf-8')

    return {
        'primary_rules_csv': primary_rules_csv,
        'primary_terms_csv': primary_terms_csv,
        'complementary_rules_csv': complementary_rules_csv,
        'complementary_terms_csv': complementary_terms_csv,
        'summary_json': summary_json,
        'if_then_txt': if_then_txt,
    }


In [ ]:
resolved_checkpoint_path = resolve_checkpoint_path(DATASET_KEY, FOLD_IDX, CHECKPOINT_PATH)
model, checkpoint_feature_names, gh_params, payload = load_gh_checkpoint(resolved_checkpoint_path, DEVICE)

base_mean_w = None
residual_mean_w = None
snapshot_error = None

if COMPUTE_MEAN_RULE_ACTIVATION:
    try:
        X_snapshot, y_snapshot, snapshot_feature_names = load_snapshot_bundle(DATASET_KEY)
        X_aligned = align_snapshot_to_checkpoint(X_snapshot, checkpoint_feature_names)
        x_tensor = apply_saved_scaler(X_aligned, payload, DEVICE)
        base_mean_w, residual_mean_w = collect_rule_activation_stats(model, x_tensor)
    except Exception as exc:
        snapshot_error = exc

checkpoint_info = {
    'checkpoint_path': resolved_checkpoint_path,
    'dataset_key': DATASET_KEY,
    'fold_idx': int(FOLD_IDX),
}

print('Resolved checkpoint:', resolved_checkpoint_path)
print('n_features:', model.n_features)
print('n_outputs:', model.n_outputs)
print('base_rules (Primary):', model.base_rules)
print('residual_rules (Complementary):', model.residual_rules)
print('mf_per_feature:', model.mf_per_feature)
print('residual_gate_mode:', model.residual_gate_mode)
print('base_mask_threshold:', model.base_mask_threshold)
print('residual_mask_threshold:', model.residual_mask_threshold)
print('feature_names[:10]:', checkpoint_feature_names[:10])

if snapshot_error is None and COMPUTE_MEAN_RULE_ACTIVATION:
    print('Mean rule activation was computed from the local dataset snapshot.')
elif COMPUTE_MEAN_RULE_ACTIVATION:
    print('Mean rule activation was skipped due to:', repr(snapshot_error))


In [ ]:
primary_rules_df, primary_terms_df = module_rule_tables(
    model,
    checkpoint_feature_names,
    module='base',
    mean_rule_weights=base_mean_w,
    top_terms=TOP_TERMS_PER_RULE,
)

complementary_rules_df, complementary_terms_df = module_rule_tables(
    model,
    checkpoint_feature_names,
    module='residual',
    mean_rule_weights=residual_mean_w,
    top_terms=TOP_TERMS_PER_RULE,
)

print('Primary rules table')
display(primary_rules_df)

print('Complementary rules table')
display(complementary_rules_df)

print('Primary term table')
display(primary_terms_df)

print('Complementary term table')
display(complementary_terms_df)


In [ ]:
print_if_then_rules(primary_rules_df, 'Primary / base')
print()
print_if_then_rules(complementary_rules_df, 'Complementary / residual')


In [ ]:
stem = f'{DATASET_KEY}_fold_{int(FOLD_IDX):02d}'
exported_paths = export_rule_outputs(
    export_dir=EXPORT_DIR,
    stem=stem,
    primary_rules_df=primary_rules_df,
    primary_terms_df=primary_terms_df,
    complementary_rules_df=complementary_rules_df,
    complementary_terms_df=complementary_terms_df,
    checkpoint_info=checkpoint_info,
)

print('Exported files:')
for key, path in exported_paths.items():
    print(f'  {key}: {path.resolve()}')


## BCWD model-interpretation cases


# BCWD Case Interpretation Across Models

- 대상 데이터셋: `Breast_Cancer_Wisconsin_(Original)`
- 비교 모델: `ANFIS`, `GA-ANFIS`, `PSO-ANFIS`, `H-ANFIS`, `GRS-ANFIS`
- 이 노트북은 **실제 malignant(유방암) 환자 1명**을 골라서 같은 케이스를 각 모델이 어떻게 해석하는지 보여줍니다.
- E404 전처리 기준에서 BCWD는 `9`개 raw numeric feature를 그대로 사용합니다.
- 따라서 아래의 환자 원시값과 모델 입력값은 동일한 `9`개 변수이며, 체크포인트에 저장된 feature 순서와 scaler만 맞춰 각 모델 규칙을 해석합니다.
- `GRS-ANFIS`는 malignant validation 케이스 중에서 `Primary(base)`는 malignant 확률이 높게 보지만 `Complementary(residual)`를 더한 뒤 `full` 예측이 가장 많이 달라지는 샘플을 우선 선택합니다.
- 기본 설정에서는 각 rule의 antecedent를 최대 `9`개 조건까지 보여주므로, BCWD에서는 사실상 전체 입력 조건을 확인할 수 있습니다.

설정은 아래 셀의 `FOLD_IDX`, `CASE_INDEX_OVERRIDE`, `H_MODEL_NAME`만 바꾸면 됩니다.



In [ ]:
from pathlib import Path
import json
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path: Path) -> bool:
    markers = ['model.py', 'data.py', 'learning.py', 'gh_config.py']
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E404',
        PROJECT_ROOT / 'GH-ANFIS_E404',
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E403',
        PROJECT_ROOT / 'GH-ANFIS_E403',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS, TSKANFIS, ParallelHierarchicalTSKANFIS
from gh_config import normalize_gh_params
from data import load_bcwd_data, coerce_numeric_frame, drop_nan_targets

PROJECT_ROOT


In [ ]:
def safe_name(name: str) -> str:
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(name))


SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_NAME = 'Breast_Cancer_Wisconsin_(Original)'
RAW_BCWD_CSV = PROJECT_ROOT / 'data' / 'bcwd_uci_15.csv'
CV_WEIGHT_ROOT = PROJECT_ROOT / 'hyper_parameter' / 'cv_weights'
EXPORT_DIR = PROJECT_ROOT / 'output' / 'bcwd_case_model_interpretation'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

FOLD_IDX = 4
CASE_INDEX_OVERRIDE = None   # 예: 137 처럼 row index를 직접 지정하면 자동 탐색 대신 그 케이스를 사용
MIN_BASE_POSITIVE_PROB = 0.80
TOP_RULES_PER_MODEL = 3
TOP_TERMS_PER_RULE = 9
H_MODEL_NAME = 'PH-ANFIS(Avg)'   # 또는 'PH-ANFIS(Stacked)'
ARTIFACT_DATASET_DIRNAME = None  # 예: 'Breast_Cancer_Wisconsin__Original___no_mi'

MODEL_FILE_STEM = {
    'GH-ANFIS': 'gh_anfis',
    'ANFIS': 'anfis',
    'GA-ANFIS': 'ga_anfis',
    'PSO-ANFIS': 'pso_anfis',
    'PH-ANFIS(Avg)': 'ph_anfis_avg',
    'PH-ANFIS(Stacked)': 'ph_anfis_stacked',
}

MODEL_NAMES = ['ANFIS', 'GA-ANFIS', 'PSO-ANFIS', H_MODEL_NAME, 'GH-ANFIS']
MODEL_DISPLAY_NAMES = {
    'ANFIS': 'ANFIS',
    'GA-ANFIS': 'GA-ANFIS',
    'PSO-ANFIS': 'PSO-ANFIS',
    'PH-ANFIS(Avg)': 'H-ANFIS',
    'PH-ANFIS(Stacked)': 'H-ANFIS(Stacked)',
    'GH-ANFIS': 'GRS-ANFIS',
}

print('DEVICE =', DEVICE)
print('RAW_BCWD_CSV =', RAW_BCWD_CSV)
print('CV_WEIGHT_ROOT =', CV_WEIGHT_ROOT)
print('FOLD_IDX =', FOLD_IDX)
print('H_MODEL_NAME =', H_MODEL_NAME)


In [ ]:
def sigmoid_np(x: np.ndarray | float) -> np.ndarray | float:
    x_clip = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x_clip))


def load_bcwd_views() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, list[int], dict[int, int]]:
    raw_df = pd.read_csv(RAW_BCWD_CSV)
    if 'target' not in raw_df.columns:
        raise ValueError(f"Raw BCWD snapshot is missing 'target': {RAW_BCWD_CSV}")

    raw_x_df = raw_df.drop(columns=['target']).reset_index(drop=True)
    y_raw = pd.to_numeric(raw_df['target'], errors='raise').astype(int).to_numpy()

    processed_x_df, y_encoded, _ = load_bcwd_data()
    processed_x_df = coerce_numeric_frame(processed_x_df)
    processed_x_df, y_encoded = drop_nan_targets(processed_x_df, y_encoded)
    processed_x_df = processed_x_df.reset_index(drop=True)
    y_encoded = np.asarray(y_encoded, dtype=int)

    if len(raw_x_df) != len(processed_x_df):
        raise ValueError(
            f'Raw snapshot rows ({len(raw_x_df)}) and processed BCWD rows ({len(processed_x_df)}) do not match.'
        )

    raw_classes = sorted(pd.unique(y_raw).tolist())
    raw_to_encoded = {int(raw): idx for idx, raw in enumerate(raw_classes)}
    y_from_raw = np.asarray([raw_to_encoded[int(v)] for v in y_raw], dtype=int)
    if not np.array_equal(y_from_raw, y_encoded):
        raise ValueError('Raw target mapping and processed label encoding are not aligned.')

    return raw_x_df, processed_x_df, y_raw, y_encoded, raw_classes, raw_to_encoded


def get_bcwd_fold_indices(y_encoded: np.ndarray, fold_idx: int, seed: int = SEED, n_splits: int = 5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(np.zeros(len(y_encoded)), y_encoded))
    if int(fold_idx) < 1 or int(fold_idx) > len(splits):
        raise ValueError(f'fold_idx must be in [1, {len(splits)}], got {fold_idx}')
    train_idx, val_idx = splits[int(fold_idx) - 1]
    return np.asarray(train_idx), np.asarray(val_idx)


def resolve_artifact_dataset_dir(
    dataset_name: str,
    fold_idx: int,
    required_models: list[str],
    root_dir: Path = CV_WEIGHT_ROOT,
    preferred_dirname: str | None = None,
) -> Path:
    if preferred_dirname is not None:
        candidate = Path(root_dir) / str(preferred_dirname)
        if not candidate.exists():
            raise FileNotFoundError(f'Preferred artifact dataset dir not found: {candidate}')
        return candidate.resolve()

    dataset_key = safe_name(dataset_name)
    seen = set()
    candidates = []
    for cand in [Path(root_dir) / dataset_key, *sorted(Path(root_dir).glob(f'{dataset_key}*'))]:
        if not cand.exists() or not cand.is_dir():
            continue
        key = str(cand.resolve())
        if key in seen:
            continue
        seen.add(key)
        candidates.append(cand)

    if not candidates:
        raise FileNotFoundError(f'No artifact directories found for dataset key prefix: {dataset_key}')

    scored = []
    fold_name = f'fold_{int(fold_idx):02d}'
    for cand in candidates:
        fold_dir = cand / fold_name
        missing = []
        if not fold_dir.exists():
            missing.append('<missing fold dir>')
        else:
            for model_name in required_models:
                stem = MODEL_FILE_STEM[model_name]
                if not (fold_dir / f'{stem}.pt').exists():
                    missing.append(f'{stem}.pt')
        score = (
            0 if cand.name.endswith('___no_mi') else 1,
            len(missing),
            cand.name,
        )
        scored.append((score, cand, missing))

    valid = [(score, cand) for score, cand, missing in scored if not missing]
    if valid:
        valid.sort(key=lambda item: item[0])
        return valid[0][1].resolve()

    detail_lines = [f'{cand.name}: missing={missing}' for _, cand, missing in scored]
    raise FileNotFoundError(
        'Could not find an artifact directory containing all requested model checkpoints.\n'
        + '\n'.join(detail_lines)
    )



def _infer_model_config(payload: dict[str, Any], params: dict[str, Any]) -> dict[str, Any]:
    cfg = payload.get('model_config')
    if cfg is not None:
        return dict(cfg)

    n_features = payload.get('n_features')
    n_outputs = payload.get('n_outputs')
    if n_features is None or n_outputs is None:
        raise KeyError('Checkpoint must contain either model_config or n_features/n_outputs.')

    out = {
        'n_features': int(n_features),
        'n_outputs': int(n_outputs),
    }
    if 'base_rules' in params:
        out['base_rules'] = int(params['base_rules'])
    if 'residual_rules' in params:
        out['residual_rules'] = int(params['residual_rules'])
    if 'mf_per_feature' in params:
        out['mf_per_feature'] = int(params['mf_per_feature'])
    if 'n_rules' in params:
        out['n_rules'] = int(params['n_rules'])
    if 'mfs_per_input' in params:
        out['mfs_per_input'] = int(params['mfs_per_input'])
    if 'branch_rules' in params:
        out['branch_rules'] = int(params['branch_rules'])
    if 'top_rules' in params:
        out['top_rules'] = int(params['top_rules'])
    return out


def apply_saved_scaler_to_array(X_arr: np.ndarray, payload: dict[str, Any]) -> np.ndarray:
    mean = payload.get('scaler_mean')
    scale = payload.get('scaler_scale')
    X_arr = np.asarray(X_arr, dtype=np.float32)
    if mean is None or scale is None:
        return X_arr.astype(np.float32)

    mean_arr = np.asarray(mean, dtype=np.float32)
    scale_arr = np.asarray(scale, dtype=np.float32)
    safe_scale = np.where(scale_arr == 0, 1.0, scale_arr)
    return ((X_arr - mean_arr) / safe_scale).astype(np.float32)


def feature_names_from_payload(payload: dict[str, Any], fallback_columns: list[str]) -> list[str]:
    feature_names = payload.get('feature_names')
    if feature_names is None:
        n_features = int(payload.get('n_features', len(fallback_columns)))
        return list(fallback_columns[:n_features])
    return list(feature_names)


def preprocess_with_payload(X_processed_df: pd.DataFrame, payload: dict[str, Any]) -> tuple[pd.DataFrame, np.ndarray]:
    feature_names = feature_names_from_payload(payload, list(X_processed_df.columns))
    missing = [col for col in feature_names if col not in X_processed_df.columns]
    if missing:
        raise KeyError(f'Missing model-input columns for artifact: {missing[:10]}')
    X_sel = X_processed_df.loc[:, feature_names].copy()
    X_scaled = apply_saved_scaler_to_array(X_sel.values, payload)
    return X_sel, X_scaled


def load_cv_artifact(
    model_name: str,
    fold_idx: int,
    artifact_dir: Path | None = None,
    device: torch.device = DEVICE,
):
    artifact_dir = Path(artifact_dir or ARTIFACT_DATASET_DIR)
    fold_dir = artifact_dir / f'fold_{int(fold_idx):02d}'
    stem = MODEL_FILE_STEM[model_name]
    payload = torch.load(fold_dir / f'{stem}.pt', map_location=device)
    if not isinstance(payload, dict):
        raise TypeError(f'Unsupported checkpoint payload type: {type(payload)}')

    raw_params = payload.get('hparams') or payload.get('params') or {}
    state_dict = payload.get('model_state_dict') or payload.get('state_dict')
    if state_dict is None:
        raise KeyError('Checkpoint is missing model_state_dict/state_dict.')

    if model_name == 'GH-ANFIS':
        params = normalize_gh_params(raw_params)
        cfg = _infer_model_config(payload, params)
        model = GH_ANFIS(
            n_features=int(cfg['n_features']),
            n_outputs=int(cfg['n_outputs']),
            residual_rules=int(cfg.get('residual_rules', params.get('residual_rules', 8))),
            base_rules=int(cfg.get('base_rules', params.get('base_rules', 4))),
            mf_per_feature=int(cfg.get('mf_per_feature', params.get('mf_per_feature', 2))),
            device=device,
            residual_gate_mode=str(params.get('residual_gate_mode', 'complement')),
            rule_init_mode=str(params.get('rule_init_mode', 'balanced')),
            rule_seed=int(params.get('rule_seed', 0)),
            firing_mode=str(params.get('firing_mode', 'htsk')),
            use_input_norm=bool(params.get('use_input_norm', False)),
            enable_residual_branch=bool(params.get('enable_residual_branch', True)),
        ).to(device)
        model.load_state_dict(state_dict)

        if params.get('base_mask_threshold') is not None:
            model.base_mask_threshold = float(params['base_mask_threshold'])
        if params.get('residual_mask_threshold') is not None:
            model.residual_mask_threshold = float(params['residual_mask_threshold'])

        model.base_mask_frozen = bool(
            payload.get('base_mask_frozen', False)
            or params.get('base_hard_epochs', 0)
            or params.get('random_role_assignment', False)
        )
        model.residual_mask_frozen = bool(
            payload.get('residual_mask_frozen', False)
            or params.get('residual_hard_epochs', 0)
            or params.get('random_role_assignment', False)
        )
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        params = dict(raw_params)
        cfg = _infer_model_config(payload, params)
        n_inputs = int(cfg.get('n_inputs', cfg.get('n_features', payload['n_features'])))
        n_outputs = int(cfg.get('n_outputs', payload['n_outputs']))
        fusion_default = 'avg' if model_name == 'PH-ANFIS(Avg)' else 'stacked'
        fusion = str(cfg.get('fusion', params.get('fusion', fusion_default)))

        extra_meta = payload.get('extra_meta') or {}
        group_a_idx = list(cfg.get('group_a_idx', extra_meta.get('group_a_idx') or []))
        group_b_idx = list(cfg.get('group_b_idx', extra_meta.get('group_b_idx') or []))
        split_seed = int(cfg.get('rule_seed', params.get('split_seed', SEED)))

        if not group_a_idx or not group_b_idx:
            rng = np.random.default_rng(split_seed)
            order = np.arange(int(n_inputs), dtype=int)
            rng.shuffle(order)
            cut = int(max(1, n_inputs // 2))
            if cut >= n_inputs:
                cut = n_inputs - 1
            group_a_idx = np.sort(order[:cut]).tolist()
            group_b_idx = np.sort(order[cut:]).tolist()

        branch_rules = int(cfg.get('branch_rules', params.get('branch_rules', params.get('n_rules', 12))))
        top_rules = int(cfg.get('top_rules', params.get('top_rules', max(2, branch_rules // 2))))
        mfs_per_input = int(cfg.get('mfs_per_input', params.get('mfs_per_input', 3)))
        rule_init_mode = str(cfg.get('rule_init_mode', params.get('rule_init_mode', 'legacy')))
        rule_seed = int(cfg.get('rule_seed', params.get('rule_seed', split_seed)))
        firing_mode = str(cfg.get('firing_mode', params.get('firing_mode', 'htsk')))

        model = ParallelHierarchicalTSKANFIS(
            n_inputs=n_inputs,
            n_outputs=n_outputs,
            group_a_idx=group_a_idx,
            group_b_idx=group_b_idx,
            branch_rules=branch_rules,
            top_rules=top_rules,
            fusion=fusion,
            mfs_per_input=mfs_per_input,
            rule_init_mode=rule_init_mode,
            rule_seed=rule_seed,
            firing_mode=firing_mode,
        ).to(device)
        model.load_state_dict(state_dict)
    else:
        params = dict(raw_params)
        cfg = _infer_model_config(payload, params)
        n_inputs = int(cfg.get('n_inputs', cfg.get('n_features', payload['n_features'])))
        n_outputs = int(cfg.get('n_outputs', payload['n_outputs']))
        n_rules = int(cfg.get('n_rules', params.get('n_rules', 30)))
        mfs_per_input = int(cfg.get('mfs_per_input', params.get('mfs_per_input', 3)))
        rule_init_mode = str(cfg.get('rule_init_mode', params.get('rule_init_mode', 'legacy')))
        rule_seed = int(cfg.get('rule_seed', params.get('rule_seed', 0)))
        firing_mode = str(cfg.get('firing_mode', params.get('firing_mode', 'htsk')))

        model = TSKANFIS(
            n_inputs=n_inputs,
            n_rules=n_rules,
            n_outputs=n_outputs,
            mfs_per_input=mfs_per_input,
            rule_init_mode=rule_init_mode,
            rule_seed=rule_seed,
            firing_mode=firing_mode,
        ).to(device)
        model.load_state_dict(state_dict)

    model.eval()
    return model, payload


RAW_X_DF, PROCESSED_X_DF, Y_RAW, Y_ENC, RAW_CLASSES, RAW_TO_ENC = load_bcwd_views()
TRAIN_IDX, VAL_IDX = get_bcwd_fold_indices(Y_ENC, FOLD_IDX)
MALIGNANT_RAW_LABEL = int(RAW_CLASSES[-1])
BENIGN_RAW_LABEL = int(RAW_CLASSES[0])
MALIGNANT_ENC_LABEL = int(RAW_TO_ENC[MALIGNANT_RAW_LABEL])
ARTIFACT_DATASET_DIR = resolve_artifact_dataset_dir(
    DATASET_NAME,
    FOLD_IDX,
    MODEL_NAMES,
    preferred_dirname=ARTIFACT_DATASET_DIRNAME,
)

print('RAW_X_DF shape =', RAW_X_DF.shape)
print('PROCESSED_X_DF shape =', PROCESSED_X_DF.shape)
print('Raw classes =', RAW_CLASSES)
print('Malignant raw label =', MALIGNANT_RAW_LABEL)
print('Malignant encoded label =', MALIGNANT_ENC_LABEL)
print('Validation fold size =', len(VAL_IDX))
print('ARTIFACT_DATASET_DIR =', ARTIFACT_DATASET_DIR)


In [ ]:
def infer_term_label(centers_1d: np.ndarray, mf_idx: int) -> tuple[str, int]:
    low_idx = int(np.argmin(centers_1d))
    high_idx = int(np.argmax(centers_1d))
    if int(mf_idx) == low_idx:
        return 'low', -1
    if int(mf_idx) == high_idx:
        return 'high', 1
    return f'mid(mf{int(mf_idx) + 1})', 0


def humanize_feature_name(feature_name: Any) -> str:
    text = str(feature_name)
    if '=' in text:
        left, right = text.split('=', 1)
        return f'{left} == {right}'
    return text


def make_term_text(feature_name: Any, term_label: str) -> str:
    readable = humanize_feature_name(feature_name)
    if '=' in str(feature_name):
        if term_label == 'high':
            return f'{readable} is active'
        if term_label == 'low':
            return f'{readable} is inactive'
        return f'{readable} is partially active'
    return f'{readable} is {term_label}'


def tsk_rule_indices(model: TSKANFIS) -> np.ndarray:
    if hasattr(model, 'rule_mf_indices') and model.rule_mf_indices is not None:
        return model.rule_mf_indices.detach().cpu().numpy().astype(int)
    selector_logits = model.rule_mf_selector_logits.detach().cpu().numpy()
    return selector_logits.argmax(axis=-1).astype(int)


def build_tsk_rule_texts(model: TSKANFIS, feature_names: list[str]) -> list[str]:
    centers = model.centers.detach().cpu().numpy()
    rule_idx = tsk_rule_indices(model)
    texts = []
    for r in range(int(model.n_rules)):
        terms = []
        for d, feat in enumerate(feature_names):
            term_label, _ = infer_term_label(centers[d], int(rule_idx[r, d]))
            terms.append(make_term_text(feat, term_label))
        texts.append(' AND '.join(terms))
    return texts


@torch.no_grad()
def predict_binary_logits(model, x_tensor: torch.Tensor, batch_size: int = 512) -> np.ndarray:
    model.eval()
    outs = []
    n = int(x_tensor.shape[0])
    for start in range(0, n, batch_size):
        logits = model(x_tensor[start:start + batch_size])
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        elif logits.dim() != 1:
            raise ValueError(f'Only binary single-logit models are supported here, got {tuple(logits.shape)}')
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


@torch.no_grad()
def analyze_tsk_sample(
    model: TSKANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    top_k: int = 3,
    label: str = 'TSK-ANFIS',
):
    model.eval()
    logits, acts = model(x_tensor_2d, return_activations=True)
    if logits.dim() == 2 and logits.size(1) == 1:
        final_logit = float(logits.detach().cpu().reshape(-1)[0])
    elif logits.dim() == 1:
        final_logit = float(logits.detach().cpu().reshape(-1)[0])
    else:
        raise ValueError(f'Only binary single-logit models are supported here, got {tuple(logits.shape)}')

    final_prob = float(sigmoid_np(final_logit))
    pred_class = int(final_prob >= 0.5)

    x_arr = x_tensor_2d.detach().cpu().numpy()[0]
    rule_weights = acts['rule_weights'].detach().cpu().numpy()[0]
    rule_firing = acts.get('rule_firing')
    if rule_firing is not None:
        rule_firing = rule_firing.detach().cpu().numpy()[0]

    consequents = model.consequents.detach().cpu().numpy()
    rule_texts = build_tsk_rule_texts(model, feature_names)

    rows = []
    for r in range(int(model.n_rules)):
        rule_logit = float(np.dot(consequents[r, 0, :-1], x_arr) + consequents[r, 0, -1])
        rule_prob = float(sigmoid_np(rule_logit))
        rule_weight = float(rule_weights[r])
        firing = float(rule_firing[r]) if rule_firing is not None else np.nan
        rows.append(
            {
                'rule_idx': int(r),
                'rule_id': f'R{r + 1}',
                'rule_weight': rule_weight,
                'rule_firing': firing,
                'rule_logit': rule_logit,
                'rule_prob_class_1': rule_prob,
                'rule_contribution_logit': float(rule_weight * rule_logit),
                'antecedent_text': rule_texts[r],
                'if_then_text': f'IF {rule_texts[r]} THEN class_1 probability = {rule_prob:.4f} (rule_logit={rule_logit:.4f})',
            }
        )

    all_rules_df = (
        pd.DataFrame(rows)
        .sort_values(['rule_weight', 'rule_contribution_logit'], ascending=[False, False])
        .reset_index(drop=True)
    )
    return {
        'model_label': label,
        'final_logit': final_logit,
        'final_prob': final_prob,
        'pred_class': pred_class,
        'top_rules_df': all_rules_df.head(int(top_k)).copy(),
        'all_rules_df': all_rules_df,
    }


@torch.no_grad()
def analyze_ph_sample(
    model: ParallelHierarchicalTSKANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    top_k: int = 3,
    label: str = 'H-ANFIS',
):
    model.eval()
    y, acts = model(x_tensor_2d, return_activations=True)
    final_logit = float(y.detach().cpu().reshape(-1)[0])
    final_prob = float(sigmoid_np(final_logit))
    pred_class = int(final_prob >= 0.5)

    xa, xb = model.split_inputs(x_tensor_2d)
    feat_a = [feature_names[i] for i in model.group_a_idx]
    feat_b = [feature_names[i] for i in model.group_b_idx]

    branch_a_analysis = analyze_tsk_sample(model.branch_a, xa, feat_a, top_k=top_k, label='Branch A')
    branch_b_analysis = analyze_tsk_sample(model.branch_b, xb, feat_b, top_k=top_k, label='Branch B')

    out = {
        'model_label': label,
        'fusion': model.fusion,
        'final_logit': final_logit,
        'final_prob': final_prob,
        'pred_class': pred_class,
        'branch_a_analysis': branch_a_analysis,
        'branch_b_analysis': branch_b_analysis,
    }

    if model.fusion == 'avg':
        fusion_weight = float(torch.sigmoid(model.fusion_logits).detach().cpu().reshape(-1)[0])
        out['fusion_text'] = (
            f'final_logit = {fusion_weight:.4f} * branch_a_logit '
            f'+ {1.0 - fusion_weight:.4f} * branch_b_logit'
        )
        out['fusion_weight'] = fusion_weight
    else:
        top_input = acts['top_input']
        top_feature_names = [f'branch_output_{idx}' for idx in range(top_input.shape[1])]
        top_analysis = analyze_tsk_sample(model.top_anfis, top_input, top_feature_names, top_k=top_k, label='Top fusion ANFIS')
        out['fusion_text'] = 'stacked top-level ANFIS combines the two branch outputs.'
        out['top_analysis'] = top_analysis

    return out


def gh_branch_masks(model: GH_ANFIS) -> dict[str, np.ndarray]:
    base_soft = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
    base_hard = (base_soft >= float(model.base_mask_threshold)).astype(float)

    residual_soft = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
    residual_hard = (residual_soft >= float(model.residual_mask_threshold)).astype(float)

    if bool(model.residual_use_complement):
        residual_effective_hard = residual_hard * (1.0 - base_hard)
    else:
        residual_effective_hard = residual_hard.copy()

    return {
        'base_soft': base_soft,
        'base_hard': base_hard,
        'residual_soft': residual_soft,
        'residual_hard': residual_hard,
        'residual_effective_hard': residual_effective_hard,
    }


def gh_module_pack(model: GH_ANFIS, module: str) -> dict[str, Any]:
    masks = gh_branch_masks(model)
    if module == 'base':
        return {
            'module': 'base',
            'module_label': 'Primary',
            'display_prefix': 'P',
            'internal_prefix': 'B',
            'centers': model.s_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.s_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.base_consequents,
            'gate_hard': masks['base_hard'],
            'n_rules': int(model.base_rules),
        }
    if module == 'residual':
        return {
            'module': 'residual',
            'module_label': 'Complementary',
            'display_prefix': 'C',
            'internal_prefix': 'R',
            'centers': model.p_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.p_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.residual_consequents,
            'gate_hard': masks['residual_effective_hard'],
            'n_rules': int(model.residual_rules),
        }
    raise ValueError("module must be 'base' or 'residual'")


@torch.no_grad()
def gh_predict_binary_logits(
    model: GH_ANFIS,
    x_tensor: torch.Tensor,
    phase: str,
    mode: str,
    batch_size: int = 512,
) -> np.ndarray:
    model.eval()
    model.set_phase(phase)
    model.set_mode(mode)
    outs = []
    n = int(x_tensor.shape[0])
    for start in range(0, n, batch_size):
        logits = model(x_tensor[start:start + batch_size])
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        elif logits.dim() != 1:
            raise ValueError(f'Only binary single-logit GH models are supported here, got {tuple(logits.shape)}')
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


def gh_module_rule_details(
    model: GH_ANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    module: str,
    acts: dict[str, Any],
    top_terms: int = 8,
) -> pd.DataFrame:
    mp = gh_module_pack(model, module)

    selector_logits = np.asarray(mp['selector_logits'], dtype=np.float64)
    selector_logits = selector_logits - selector_logits.max(axis=-1, keepdims=True)
    selector_probs = np.exp(selector_logits)
    selector_probs = selector_probs / selector_probs.sum(axis=-1, keepdims=True)

    centers = np.asarray(mp['centers'], dtype=np.float64)
    x_arr = x_tensor_2d.detach().cpu().numpy()[0]

    if module == 'base':
        gate_vec = acts['gate_base'][0].detach().cpu().numpy()
        rule_weights = acts['base_rule_weights'][0].detach().cpu().numpy()
    else:
        gate_vec = acts['gate_residual'][0].detach().cpu().numpy()
        rule_weights = acts['residual_rule_weights'][0].detach().cpu().numpy()

    rows = []
    for r in range(mp['n_rules']):
        best_mf = selector_probs[r].argmax(axis=1)
        best_prob = selector_probs[r].max(axis=1)

        active_idx = np.where(gate_vec > 1e-8)[0].tolist()
        if active_idx:
            active_idx = sorted(active_idx, key=lambda d: float(best_prob[d]), reverse=True)
        else:
            active_idx = np.argsort(best_prob)[::-1].tolist()
        active_idx = active_idx[: max(1, int(top_terms))]

        terms = []
        for d in active_idx:
            term_label, _ = infer_term_label(centers[d], int(best_mf[d]))
            terms.append(make_term_text(feature_names[d], term_label))
        antecedent_text = ' AND '.join(terms) if terms else '(no active terms)'

        layer = mp['consequents'][r]
        weight = layer.weight.detach().cpu().numpy().reshape(-1)
        bias = float(layer.bias.detach().cpu().numpy().reshape(-1)[0]) if layer.bias is not None else 0.0
        feat_weight = weight[: len(feature_names)]
        bias_weight = float(weight[len(feature_names)]) if len(weight) > len(feature_names) else 0.0
        rule_logit = float(np.sum(x_arr * feat_weight * gate_vec) + bias_weight + bias)
        rule_prob = float(sigmoid_np(rule_logit))
        rule_weight = float(rule_weights[r])

        rows.append(
            {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'rule_weight': rule_weight,
                'rule_logit': rule_logit,
                'rule_prob_class_1': rule_prob,
                'rule_contribution_logit': float(rule_weight * rule_logit),
                'antecedent_text': antecedent_text,
                'if_then_text': f'IF {antecedent_text} THEN class_1 probability = {rule_prob:.4f} (rule_logit={rule_logit:.4f})',
            }
        )

    return (
        pd.DataFrame(rows)
        .sort_values(['rule_weight', 'rule_contribution_logit'], ascending=[False, False])
        .reset_index(drop=True)
    )


@torch.no_grad()
def analyze_gh_sample(
    model: GH_ANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    top_k_rules: int = 3,
    top_terms: int = 8,
    label: str = 'GRS-ANFIS',
):
    base_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='base', mode='base_only', batch_size=1)[0])
    residual_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='residual_complement', mode='residual_only', batch_size=1)[0])
    full_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='residual_complement', mode='full', batch_size=1)[0])

    base_prob = float(sigmoid_np(base_logit))
    residual_prob = float(sigmoid_np(residual_logit))
    full_prob = float(sigmoid_np(full_logit))

    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')
    _, acts = model(x_tensor_2d, return_activations=True, use_soft_eval=False)

    base_rules_df = gh_module_rule_details(model, x_tensor_2d, feature_names, module='base', acts=acts, top_terms=top_terms)
    residual_rules_df = gh_module_rule_details(model, x_tensor_2d, feature_names, module='residual', acts=acts, top_terms=top_terms)

    return {
        'model_label': label,
        'base_logit': base_logit,
        'base_prob': base_prob,
        'base_pred': int(base_prob >= 0.5),
        'residual_logit': residual_logit,
        'residual_prob': residual_prob,
        'residual_pred': int(residual_prob >= 0.5),
        'full_logit': full_logit,
        'full_prob': full_prob,
        'full_pred': int(full_prob >= 0.5),
        'delta_full_minus_base': float(full_prob - base_prob),
        'delta_residual_logit': float(full_logit - base_logit),
        'base_rules_df': base_rules_df.head(int(top_k_rules)).copy(),
        'residual_rules_df': residual_rules_df.head(int(top_k_rules)).copy(),
        'all_base_rules_df': base_rules_df,
        'all_residual_rules_df': residual_rules_df,
    }


def find_interesting_gh_case(
    model: GH_ANFIS,
    payload: dict[str, Any],
    X_processed_df: pd.DataFrame,
    y_raw: np.ndarray,
    y_encoded: np.ndarray,
    val_idx: np.ndarray,
    min_base_positive_prob: float = 0.8,
):
    X_val_processed = X_processed_df.iloc[val_idx].copy()
    _, X_val_scaled = preprocess_with_payload(X_val_processed, payload)
    x_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=DEVICE)

    base_logits = gh_predict_binary_logits(model, x_tensor, phase='base', mode='base_only')
    residual_logits = gh_predict_binary_logits(model, x_tensor, phase='residual_complement', mode='residual_only')
    full_logits = gh_predict_binary_logits(model, x_tensor, phase='residual_complement', mode='full')

    df = pd.DataFrame(
        {
            'orig_index': val_idx,
            'y_raw': y_raw[val_idx],
            'y_encoded': y_encoded[val_idx],
            'base_prob': sigmoid_np(base_logits),
            'residual_prob': sigmoid_np(residual_logits),
            'full_prob': sigmoid_np(full_logits),
            'base_logit': base_logits,
            'residual_logit': residual_logits,
            'full_logit': full_logits,
        }
    )
    df['base_pred'] = (df['base_prob'] >= 0.5).astype(int)
    df['residual_pred'] = (df['residual_prob'] >= 0.5).astype(int)
    df['full_pred'] = (df['full_prob'] >= 0.5).astype(int)
    df['delta_full_minus_base'] = df['full_prob'] - df['base_prob']
    df['prediction_flip'] = df['base_pred'] != df['full_pred']

    positive_df = df[df['y_encoded'] == MALIGNANT_ENC_LABEL].copy()
    if positive_df.empty:
        raise ValueError('No malignant validation cases were found for this fold.')

    flip_df = positive_df[
        (positive_df['base_pred'] == MALIGNANT_ENC_LABEL)
        & (positive_df['full_pred'] != positive_df['base_pred'])
    ].copy()

    if not flip_df.empty:
        selected = flip_df.sort_values(['full_prob', 'delta_full_minus_base'], ascending=[True, True]).iloc[0]
        reason = 'base는 malignant로 보지만 full에서는 예측이 뒤집히는 malignant 케이스'
    else:
        strong_df = positive_df[positive_df['base_prob'] >= float(min_base_positive_prob)].copy()
        if strong_df.empty:
            strong_df = positive_df.copy()
        selected = strong_df.sort_values(['delta_full_minus_base', 'full_prob'], ascending=[True, True]).iloc[0]
        reason = 'base 대비 full malignant 확률 하락폭이 가장 큰 malignant 케이스'

    candidate_table = positive_df.sort_values(
        ['prediction_flip', 'delta_full_minus_base', 'full_prob'],
        ascending=[False, True, True],
    ).reset_index(drop=True)
    return candidate_table, int(selected['orig_index']), reason


def describe_gh_case(analysis: dict[str, Any]) -> str:
    base_prob = float(analysis['base_prob'])
    residual_prob = float(analysis['residual_prob'])
    full_prob = float(analysis['full_prob'])
    delta = float(analysis['delta_full_minus_base'])

    if analysis['base_pred'] != analysis['full_pred']:
        direction = (
            f'Primary(base)는 class_{analysis["base_pred"]}로 보지만 '
            f'Complementary를 합친 full은 class_{analysis["full_pred"]}로 뒤집힙니다.'
        )
    elif delta < 0:
        direction = f'Complementary를 합치면 class_1 probability가 {base_prob:.4f} -> {full_prob:.4f}로 낮아집니다.'
    elif delta > 0:
        direction = f'Complementary를 합치면 class_1 probability가 {base_prob:.4f} -> {full_prob:.4f}로 높아집니다.'
    else:
        direction = f'Complementary를 합쳐도 class_1 probability는 {full_prob:.4f}로 거의 변하지 않습니다.'

    base_rule = analysis['base_rules_df'].iloc[0]['if_then_text'] if not analysis['base_rules_df'].empty else '(no primary rule)'
    residual_rule = analysis['residual_rules_df'].iloc[0]['if_then_text'] if not analysis['residual_rules_df'].empty else '(no complementary rule)'

    return (
        f'{direction} '
        f'base_prob={base_prob:.4f}, residual_prob={residual_prob:.4f}, full_prob={full_prob:.4f}. '
        f'Top Primary rule: {base_rule} '
        f'Top Complementary rule: {residual_rule}'
    )


def build_model_input_view(sample_processed_df: pd.DataFrame) -> pd.DataFrame:
    row = sample_processed_df.iloc[0]
    active = row[row != 0].sort_values(ascending=False).reset_index()
    active.columns = ['feature', 'value']
    active['feature_readable'] = active['feature'].map(humanize_feature_name)
    return active[['feature_readable', 'feature', 'value']]


In [ ]:
gh_model, gh_payload = load_cv_artifact('GH-ANFIS', FOLD_IDX)
case_candidates_df, selected_case_index, case_selection_reason = find_interesting_gh_case(
    gh_model,
    gh_payload,
    PROCESSED_X_DF,
    Y_RAW,
    Y_ENC,
    VAL_IDX,
    min_base_positive_prob=MIN_BASE_POSITIVE_PROB,
)

if CASE_INDEX_OVERRIDE is not None:
    selected_case_index = int(CASE_INDEX_OVERRIDE)
    case_selection_reason = 'manual override'

selected_case_raw = RAW_X_DF.iloc[[selected_case_index]].copy()
selected_case_processed = PROCESSED_X_DF.iloc[[selected_case_index]].copy()
selected_case_input_view = build_model_input_view(selected_case_processed)
selected_case_target_raw = int(Y_RAW[selected_case_index])
selected_case_target_encoded = int(Y_ENC[selected_case_index])
selected_case_is_val = bool(int(selected_case_index) in set(VAL_IDX.tolist()))

print('Selected case index:', selected_case_index)
print('Selection reason:', case_selection_reason)
print('Raw target:', selected_case_target_raw)
print('Encoded target:', selected_case_target_encoded)
print('Is in validation fold:', selected_case_is_val)
print('Malignant raw label:', MALIGNANT_RAW_LABEL)
print('Benign raw label:', BENIGN_RAW_LABEL)
print('Model input feature count:', len(selected_case_input_view))

print('Top candidate cases (malignant validation cases only):')
display(case_candidates_df.head(20))

print('Selected patient raw feature values (9 original variables):')
display(selected_case_raw.T.rename(columns={selected_case_index: 'value'}))

print('Selected patient model input feature values (before scaling):')
display(selected_case_input_view.head(9))


In [ ]:
analysis_by_model = {}
summary_rows = []

for model_name in MODEL_NAMES:
    model, payload = load_cv_artifact(model_name, FOLD_IDX)
    sample_feature_df, sample_scaled = preprocess_with_payload(selected_case_processed, payload)
    sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32, device=DEVICE)

    if model_name == 'GH-ANFIS':
        analysis = analyze_gh_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k_rules=TOP_RULES_PER_MODEL,
            top_terms=TOP_TERMS_PER_RULE,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['full_pred']),
                'class_1_probability': float(analysis['full_prob']),
                'base_probability': float(analysis['base_prob']),
                'residual_probability': float(analysis['residual_prob']),
                'delta_full_minus_base': float(analysis['delta_full_minus_base']),
                'top_rule_summary': (
                    f"base={analysis['base_rules_df'].iloc[0]['display_rule_id'] if not analysis['base_rules_df'].empty else 'NA'}; "
                    f"residual={analysis['residual_rules_df'].iloc[0]['display_rule_id'] if not analysis['residual_rules_df'].empty else 'NA'}"
                ),
            }
        )
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        analysis = analyze_ph_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k=TOP_RULES_PER_MODEL,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        top_rule_summary = 'branch_a=' + analysis['branch_a_analysis']['top_rules_df'].iloc[0]['rule_id']
        top_rule_summary += '; branch_b=' + analysis['branch_b_analysis']['top_rules_df'].iloc[0]['rule_id']
        if 'top_analysis' in analysis:
            top_rule_summary += '; top=' + analysis['top_analysis']['top_rules_df'].iloc[0]['rule_id']

        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['pred_class']),
                'class_1_probability': float(analysis['final_prob']),
                'base_probability': np.nan,
                'residual_probability': np.nan,
                'delta_full_minus_base': np.nan,
                'top_rule_summary': top_rule_summary,
            }
        )
    else:
        analysis = analyze_tsk_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k=TOP_RULES_PER_MODEL,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['pred_class']),
                'class_1_probability': float(analysis['final_prob']),
                'base_probability': np.nan,
                'residual_probability': np.nan,
                'delta_full_minus_base': np.nan,
                'top_rule_summary': analysis['top_rules_df'].iloc[0]['rule_id'],
            }
        )

    analysis['payload'] = payload
    analysis['sample_feature_df'] = sample_feature_df
    analysis_by_model[model_name] = analysis

comparison_df = pd.DataFrame(summary_rows)
comparison_df['true_encoded_class'] = selected_case_target_encoded
comparison_df['true_raw_target'] = selected_case_target_raw
comparison_df['correct'] = comparison_df['predicted_class'] == selected_case_target_encoded
comparison_df['sort_key'] = comparison_df['artifact_name'].map({name: idx for idx, name in enumerate(MODEL_NAMES)})
comparison_df = comparison_df.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

print('Model comparison summary for the selected malignant case:')
display(comparison_df)


In [ ]:
gh_analysis = analysis_by_model['GH-ANFIS']
gh_narrative = describe_gh_case(gh_analysis)

print('GRS-focused interpretation:')
print(gh_narrative)

for model_name in MODEL_NAMES:
    analysis = analysis_by_model[model_name]
    display_name = MODEL_DISPLAY_NAMES[model_name]

    print('=' * 100)
    print(f'[{display_name}] artifact={model_name}')

    if model_name == 'GH-ANFIS':
        print(f"base_prob={analysis['base_prob']:.4f}, residual_prob={analysis['residual_prob']:.4f}, full_prob={analysis['full_prob']:.4f}")
        print(f"delta_full_minus_base={analysis['delta_full_minus_base']:.4f}, full_pred=class_{analysis['full_pred']}")
        print('[Top Primary/base rules]')
        display(analysis['base_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        print('[Top Complementary/residual rules]')
        display(analysis['residual_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        print(f"final_prob={analysis['final_prob']:.4f}, pred=class_{analysis['pred_class']}")
        print('fusion:', analysis['fusion_text'])
        print('[Branch A top rules]')
        display(analysis['branch_a_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        print('[Branch B top rules]')
        display(analysis['branch_b_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        if 'top_analysis' in analysis:
            print('[Top fusion ANFIS rules]')
            display(analysis['top_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
    else:
        print(f"final_prob={analysis['final_prob']:.4f}, pred=class_{analysis['pred_class']}")
        display(analysis['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])


In [ ]:
def df_to_records(df: pd.DataFrame) -> list[dict[str, Any]]:
    if df is None or df.empty:
        return []
    return json.loads(df.to_json(orient='records', force_ascii=False))


stem = f'bcwd_fold_{int(FOLD_IDX):02d}_case_{int(selected_case_index):03d}'
comparison_csv = EXPORT_DIR / f'{stem}_comparison.csv'
candidates_csv = EXPORT_DIR / f'{stem}_gh_candidates.csv'
selected_case_raw_csv = EXPORT_DIR / f'{stem}_raw_features.csv'
selected_case_input_view_csv = EXPORT_DIR / f'{stem}_model_input_features.csv'
top_rules_json = EXPORT_DIR / f'{stem}_top_rules.json'
summary_json = EXPORT_DIR / f'{stem}_summary.json'

comparison_df.to_csv(comparison_csv, index=False)
case_candidates_df.to_csv(candidates_csv, index=False)
selected_case_raw.T.rename(columns={selected_case_index: 'value'}).to_csv(selected_case_raw_csv)
selected_case_input_view.to_csv(selected_case_input_view_csv, index=False)

rules_payload = {}
for model_name in MODEL_NAMES:
    analysis = analysis_by_model[model_name]
    display_name = MODEL_DISPLAY_NAMES[model_name]
    if model_name == 'GH-ANFIS':
        rules_payload[display_name] = {
            'base_rules': df_to_records(analysis['base_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
            'residual_rules': df_to_records(analysis['residual_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        payload = {
            'branch_a_rules': df_to_records(analysis['branch_a_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
            'branch_b_rules': df_to_records(analysis['branch_b_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }
        if 'top_analysis' in analysis:
            payload['top_fusion_rules'] = df_to_records(analysis['top_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        rules_payload[display_name] = payload
    else:
        rules_payload[display_name] = {
            'rules': df_to_records(analysis['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }

top_rules_json.write_text(json.dumps(rules_payload, ensure_ascii=False, indent=2), encoding='utf-8')

summary_payload = {
    'dataset_name': DATASET_NAME,
    'artifact_dataset_dir': str(ARTIFACT_DATASET_DIR),
    'fold_idx': int(FOLD_IDX),
    'selected_case_index': int(selected_case_index),
    'selection_reason': case_selection_reason,
    'selected_case_target_raw': int(selected_case_target_raw),
    'selected_case_target_encoded': int(selected_case_target_encoded),
    'is_validation_case': bool(selected_case_is_val),
    'label_mapping_raw_to_encoded': {str(k): int(v) for k, v in RAW_TO_ENC.items()},
    'gh_narrative': gh_narrative,
    'models': {
        row['artifact_name']: {
            'probability_class_1': float(row['class_1_probability']),
            'predicted_class': int(row['predicted_class']),
            'correct': bool(row['correct']),
        }
        for _, row in comparison_df.iterrows()
    },
}
summary_json.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('Exported:')
print('  comparison_csv =', comparison_csv.resolve())
print('  candidates_csv =', candidates_csv.resolve())
print('  selected_case_raw_csv =', selected_case_raw_csv.resolve())
print('  selected_case_input_view_csv =', selected_case_input_view_csv.resolve())
print('  top_rules_json =', top_rules_json.resolve())
print('  summary_json =', summary_json.resolve())


## BCWD base-to-full recovery case


# BCWD GH Recovery Case Interpretation

- 대상 데이터셋: `Breast_Cancer_Wisconsin_(Original)`
- 비교 모델: `ANFIS`, `GA-ANFIS`, `PSO-ANFIS`, `H-ANFIS`, `GRS-ANFIS`
- 이 노트북은 BCWD 5-fold validation 전체를 검색해서 다음 조건을 만족하는 사례를 자동으로 찾습니다.
  - `ground truth = malignant`
  - `GH-ANFIS Primary(base) = benign`
  - `GH-ANFIS full = malignant`
- 찾은 사례 1개에 대해 각 모델의 예측 확률과 top fired fuzzy rule을 `IF ... THEN ...` 형식으로 비교합니다.
- E404 전처리 기준에서 BCWD는 `9`개 raw numeric feature를 그대로 사용합니다.
- 따라서 이 노트북에서 보여주는 `원시 검사값`과 `모델 입력 특징`은 동일한 9개 변수이며, 체크포인트의 feature 순서와 scaler만 다시 맞춰 사용합니다.
- 기본 설정에서는 각 rule의 antecedent를 최대 `9`개 조건까지 보여주므로, BCWD에서는 사실상 전체 입력 조건을 확인할 수 있습니다.

설정은 아래 셀의 `SEARCH_FOLDS`, `H_MODEL_NAME`, `FOLD_IDX_OVERRIDE`, `CASE_INDEX_OVERRIDE`를 바꾸면 됩니다.



In [ ]:
from pathlib import Path
import json
import os
import sys
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd().resolve()


def _looks_like_project_root(path: Path) -> bool:
    markers = ['model.py', 'data.py', 'learning.py', 'gh_config.py']
    return all((path / marker).exists() for marker in markers)


if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not _looks_like_project_root(PROJECT_ROOT):
    candidate_roots = [
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E404',
        PROJECT_ROOT / 'GH-ANFIS_E404',
        PROJECT_ROOT / '03_Research' / 'GH-ANFIS_E403',
        PROJECT_ROOT / 'GH-ANFIS_E403',
    ]
    for cand in candidate_roots:
        if _looks_like_project_root(cand):
            PROJECT_ROOT = cand
            break
    else:
        for parent in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
            if _looks_like_project_root(parent):
                PROJECT_ROOT = parent
                break

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model import GH_ANFIS, TSKANFIS, ParallelHierarchicalTSKANFIS
from gh_config import normalize_gh_params
from data import load_bcwd_data, coerce_numeric_frame, drop_nan_targets

PROJECT_ROOT


In [ ]:
def safe_name(name: str) -> str:
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(name))


SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATASET_NAME = 'Breast_Cancer_Wisconsin_(Original)'
RAW_BCWD_CSV = PROJECT_ROOT / 'data' / 'bcwd_uci_15.csv'
CV_WEIGHT_ROOT = PROJECT_ROOT / 'hyper_parameter' / 'cv_weights'
EXPORT_DIR = PROJECT_ROOT / 'output' / 'bcwd_gh_recovery_case_interpretation'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SEARCH_FOLDS = [1, 2, 3, 4, 5]
FOLD_IDX_OVERRIDE = None
CASE_INDEX_OVERRIDE = None
TOP_RULES_PER_MODEL = 3
TOP_TERMS_PER_RULE = 9
H_MODEL_NAME = 'PH-ANFIS(Avg)'
ARTIFACT_DATASET_DIRNAME = None

MODEL_FILE_STEM = {
    'GH-ANFIS': 'gh_anfis',
    'ANFIS': 'anfis',
    'GA-ANFIS': 'ga_anfis',
    'PSO-ANFIS': 'pso_anfis',
    'PH-ANFIS(Avg)': 'ph_anfis_avg',
    'PH-ANFIS(Stacked)': 'ph_anfis_stacked',
}

MODEL_NAMES = ['ANFIS', 'GA-ANFIS', 'PSO-ANFIS', H_MODEL_NAME, 'GH-ANFIS']
MODEL_DISPLAY_NAMES = {
    'ANFIS': 'ANFIS',
    'GA-ANFIS': 'GA-ANFIS',
    'PSO-ANFIS': 'PSO-ANFIS',
    'PH-ANFIS(Avg)': 'H-ANFIS',
    'PH-ANFIS(Stacked)': 'H-ANFIS(Stacked)',
    'GH-ANFIS': 'GRS-ANFIS',
}

print('DEVICE =', DEVICE)
print('RAW_BCWD_CSV =', RAW_BCWD_CSV)
print('CV_WEIGHT_ROOT =', CV_WEIGHT_ROOT)
print('SEARCH_FOLDS =', SEARCH_FOLDS)
print('H_MODEL_NAME =', H_MODEL_NAME)


In [ ]:
def sigmoid_np(x: np.ndarray | float) -> np.ndarray | float:
    x_clip = np.clip(x, -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-x_clip))


def load_bcwd_views() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, list[int], dict[int, int]]:
    raw_df = pd.read_csv(RAW_BCWD_CSV)
    if 'target' not in raw_df.columns:
        raise ValueError(f"Raw BCWD snapshot is missing 'target': {RAW_BCWD_CSV}")

    raw_x_df = raw_df.drop(columns=['target']).reset_index(drop=True)
    y_raw = pd.to_numeric(raw_df['target'], errors='raise').astype(int).to_numpy()

    processed_x_df, y_encoded, _ = load_bcwd_data()
    processed_x_df = coerce_numeric_frame(processed_x_df)
    processed_x_df, y_encoded = drop_nan_targets(processed_x_df, y_encoded)
    processed_x_df = processed_x_df.reset_index(drop=True)
    y_encoded = np.asarray(y_encoded, dtype=int)

    if len(raw_x_df) != len(processed_x_df):
        raise ValueError('Raw BCWD rows and processed BCWD rows do not match.')

    raw_classes = sorted(pd.unique(y_raw).tolist())
    raw_to_encoded = {int(raw): idx for idx, raw in enumerate(raw_classes)}
    y_from_raw = np.asarray([raw_to_encoded[int(v)] for v in y_raw], dtype=int)
    if not np.array_equal(y_from_raw, y_encoded):
        raise ValueError('Raw target mapping and processed label encoding are not aligned.')

    return raw_x_df, processed_x_df, y_raw, y_encoded, raw_classes, raw_to_encoded


def get_bcwd_fold_indices(y_encoded: np.ndarray, fold_idx: int, seed: int = SEED, n_splits: int = 5):
    splitter = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    splits = list(splitter.split(np.zeros(len(y_encoded)), y_encoded))
    if int(fold_idx) < 1 or int(fold_idx) > len(splits):
        raise ValueError(f'fold_idx must be in [1, {len(splits)}], got {fold_idx}')
    train_idx, val_idx = splits[int(fold_idx) - 1]
    return np.asarray(train_idx), np.asarray(val_idx)


def resolve_artifact_dataset_dir(
    dataset_name: str,
    fold_idx: int,
    required_models: list[str],
    root_dir: Path = CV_WEIGHT_ROOT,
    preferred_dirname: str | None = None,
) -> Path:
    if preferred_dirname is not None:
        candidate = Path(root_dir) / str(preferred_dirname)
        if not candidate.exists():
            raise FileNotFoundError(f'Preferred artifact dataset dir not found: {candidate}')
        return candidate.resolve()

    dataset_key = safe_name(dataset_name)
    seen = set()
    candidates = []
    for cand in [Path(root_dir) / dataset_key, *sorted(Path(root_dir).glob(f'{dataset_key}*'))]:
        if not cand.exists() or not cand.is_dir():
            continue
        key = str(cand.resolve())
        if key in seen:
            continue
        seen.add(key)
        candidates.append(cand)

    if not candidates:
        raise FileNotFoundError(f'No artifact directories found for dataset key prefix: {dataset_key}')

    scored = []
    fold_name = f'fold_{int(fold_idx):02d}'
    for cand in candidates:
        fold_dir = cand / fold_name
        missing = []
        if not fold_dir.exists():
            missing.append('<missing fold dir>')
        else:
            for model_name in required_models:
                stem = MODEL_FILE_STEM[model_name]
                if not (fold_dir / f'{stem}.pt').exists():
                    missing.append(f'{stem}.pt')
        score = (0 if cand.name.endswith('___no_mi') else 1, len(missing), cand.name)
        scored.append((score, cand, missing))

    valid = [(score, cand) for score, cand, missing in scored if not missing]
    if valid:
        valid.sort(key=lambda item: item[0])
        return valid[0][1].resolve()

    detail = '; '.join([f'{cand.name}: {missing}' for _, cand, missing in scored])
    raise FileNotFoundError('Could not find artifact directory for requested models. ' + detail)


def _infer_model_config(payload: dict[str, Any], params: dict[str, Any]) -> dict[str, Any]:
    cfg = payload.get('model_config')
    if cfg is not None:
        return dict(cfg)

    n_features = payload.get('n_features')
    n_outputs = payload.get('n_outputs')
    if n_features is None or n_outputs is None:
        raise KeyError('Checkpoint must contain either model_config or n_features/n_outputs.')

    out = {'n_features': int(n_features), 'n_outputs': int(n_outputs)}
    for key in ['base_rules', 'residual_rules', 'mf_per_feature', 'n_rules', 'mfs_per_input', 'branch_rules', 'top_rules']:
        if key in params:
            out[key] = int(params[key])
    return out


def apply_saved_scaler_to_array(X_arr: np.ndarray, payload: dict[str, Any]) -> np.ndarray:
    mean = payload.get('scaler_mean')
    scale = payload.get('scaler_scale')
    X_arr = np.asarray(X_arr, dtype=np.float32)
    if mean is None or scale is None:
        return X_arr.astype(np.float32)

    mean_arr = np.asarray(mean, dtype=np.float32)
    scale_arr = np.asarray(scale, dtype=np.float32)
    safe_scale = np.where(scale_arr == 0, 1.0, scale_arr)
    return ((X_arr - mean_arr) / safe_scale).astype(np.float32)


def feature_names_from_payload(payload: dict[str, Any], fallback_columns: list[str]) -> list[str]:
    feature_names = payload.get('feature_names')
    if feature_names is None:
        n_features = int(payload.get('n_features', len(fallback_columns)))
        return list(fallback_columns[:n_features])
    return list(feature_names)


def preprocess_with_payload(X_processed_df: pd.DataFrame, payload: dict[str, Any]) -> tuple[pd.DataFrame, np.ndarray]:
    feature_names = feature_names_from_payload(payload, list(X_processed_df.columns))
    missing = [col for col in feature_names if col not in X_processed_df.columns]
    if missing:
        raise KeyError(f'Missing model-input columns for artifact: {missing[:10]}')
    X_sel = X_processed_df.loc[:, feature_names].copy()
    X_scaled = apply_saved_scaler_to_array(X_sel.values, payload)
    return X_sel, X_scaled


def load_cv_artifact(model_name: str, fold_idx: int, artifact_dir: Path, device: torch.device = DEVICE):
    artifact_dir = Path(artifact_dir)
    fold_dir = artifact_dir / f'fold_{int(fold_idx):02d}'
    stem = MODEL_FILE_STEM[model_name]
    payload = torch.load(fold_dir / f'{stem}.pt', map_location=device)
    if not isinstance(payload, dict):
        raise TypeError(f'Unsupported checkpoint payload type: {type(payload)}')

    raw_params = payload.get('hparams') or payload.get('params') or {}
    state_dict = payload.get('model_state_dict') or payload.get('state_dict')
    if state_dict is None:
        raise KeyError('Checkpoint is missing model_state_dict/state_dict.')

    if model_name == 'GH-ANFIS':
        params = normalize_gh_params(raw_params)
        cfg = _infer_model_config(payload, params)
        model = GH_ANFIS(
            n_features=int(cfg['n_features']),
            n_outputs=int(cfg['n_outputs']),
            residual_rules=int(cfg.get('residual_rules', params.get('residual_rules', 8))),
            base_rules=int(cfg.get('base_rules', params.get('base_rules', 4))),
            mf_per_feature=int(cfg.get('mf_per_feature', params.get('mf_per_feature', 2))),
            device=device,
            residual_gate_mode=str(params.get('residual_gate_mode', 'complement')),
            rule_init_mode=str(params.get('rule_init_mode', 'balanced')),
            rule_seed=int(params.get('rule_seed', 0)),
            firing_mode=str(params.get('firing_mode', 'htsk')),
            use_input_norm=bool(params.get('use_input_norm', False)),
            enable_residual_branch=bool(params.get('enable_residual_branch', True)),
        ).to(device)
        model.load_state_dict(state_dict)

        if params.get('base_mask_threshold') is not None:
            model.base_mask_threshold = float(params['base_mask_threshold'])
        if params.get('residual_mask_threshold') is not None:
            model.residual_mask_threshold = float(params['residual_mask_threshold'])

        model.base_mask_frozen = bool(
            payload.get('base_mask_frozen', False)
            or params.get('base_hard_epochs', 0)
            or params.get('random_role_assignment', False)
        )
        model.residual_mask_frozen = bool(
            payload.get('residual_mask_frozen', False)
            or params.get('residual_hard_epochs', 0)
            or params.get('random_role_assignment', False)
        )
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        params = dict(raw_params)
        cfg = _infer_model_config(payload, params)
        n_inputs = int(cfg.get('n_inputs', cfg.get('n_features', payload['n_features'])))
        n_outputs = int(cfg.get('n_outputs', payload['n_outputs']))
        fusion_default = 'avg' if model_name == 'PH-ANFIS(Avg)' else 'stacked'
        fusion = str(cfg.get('fusion', params.get('fusion', fusion_default)))

        extra_meta = payload.get('extra_meta') or {}
        group_a_idx = list(cfg.get('group_a_idx', extra_meta.get('group_a_idx') or []))
        group_b_idx = list(cfg.get('group_b_idx', extra_meta.get('group_b_idx') or []))
        split_seed = int(cfg.get('rule_seed', params.get('split_seed', SEED)))
        if not group_a_idx or not group_b_idx:
            rng = np.random.default_rng(split_seed)
            order = np.arange(int(n_inputs), dtype=int)
            rng.shuffle(order)
            cut = int(max(1, n_inputs // 2))
            if cut >= n_inputs:
                cut = n_inputs - 1
            group_a_idx = np.sort(order[:cut]).tolist()
            group_b_idx = np.sort(order[cut:]).tolist()

        branch_rules = int(cfg.get('branch_rules', params.get('branch_rules', params.get('n_rules', 12))))
        top_rules = int(cfg.get('top_rules', params.get('top_rules', max(2, branch_rules // 2))))
        mfs_per_input = int(cfg.get('mfs_per_input', params.get('mfs_per_input', 3)))
        rule_init_mode = str(cfg.get('rule_init_mode', params.get('rule_init_mode', 'legacy')))
        rule_seed = int(cfg.get('rule_seed', params.get('rule_seed', split_seed)))
        firing_mode = str(cfg.get('firing_mode', params.get('firing_mode', 'htsk')))

        model = ParallelHierarchicalTSKANFIS(
            n_inputs=n_inputs,
            n_outputs=n_outputs,
            group_a_idx=group_a_idx,
            group_b_idx=group_b_idx,
            branch_rules=branch_rules,
            top_rules=top_rules,
            fusion=fusion,
            mfs_per_input=mfs_per_input,
            rule_init_mode=rule_init_mode,
            rule_seed=rule_seed,
            firing_mode=firing_mode,
        ).to(device)
        model.load_state_dict(state_dict)
    else:
        params = dict(raw_params)
        cfg = _infer_model_config(payload, params)
        n_inputs = int(cfg.get('n_inputs', cfg.get('n_features', payload['n_features'])))
        n_outputs = int(cfg.get('n_outputs', payload['n_outputs']))
        n_rules = int(cfg.get('n_rules', params.get('n_rules', 30)))
        mfs_per_input = int(cfg.get('mfs_per_input', params.get('mfs_per_input', 3)))
        rule_init_mode = str(cfg.get('rule_init_mode', params.get('rule_init_mode', 'legacy')))
        rule_seed = int(cfg.get('rule_seed', params.get('rule_seed', 0)))
        firing_mode = str(cfg.get('firing_mode', params.get('firing_mode', 'htsk')))

        model = TSKANFIS(
            n_inputs=n_inputs,
            n_rules=n_rules,
            n_outputs=n_outputs,
            mfs_per_input=mfs_per_input,
            rule_init_mode=rule_init_mode,
            rule_seed=rule_seed,
            firing_mode=firing_mode,
        ).to(device)
        model.load_state_dict(state_dict)

    model.eval()
    return model, payload


RAW_X_DF, PROCESSED_X_DF, Y_RAW, Y_ENC, RAW_CLASSES, RAW_TO_ENC = load_bcwd_views()
MALIGNANT_RAW_LABEL = int(RAW_CLASSES[-1])
BENIGN_RAW_LABEL = int(RAW_CLASSES[0])
MALIGNANT_ENC_LABEL = int(RAW_TO_ENC[MALIGNANT_RAW_LABEL])

print('RAW_X_DF shape =', RAW_X_DF.shape)
print('PROCESSED_X_DF shape =', PROCESSED_X_DF.shape)
print('Raw classes =', RAW_CLASSES)
print('Malignant raw label =', MALIGNANT_RAW_LABEL)
print('Malignant encoded label =', MALIGNANT_ENC_LABEL)


In [ ]:
def infer_term_label(centers_1d: np.ndarray, mf_idx: int) -> tuple[str, int]:
    low_idx = int(np.argmin(centers_1d))
    high_idx = int(np.argmax(centers_1d))
    if int(mf_idx) == low_idx:
        return 'low', -1
    if int(mf_idx) == high_idx:
        return 'high', 1
    return f'mid(mf{int(mf_idx) + 1})', 0


def humanize_feature_name(feature_name: Any) -> str:
    text = str(feature_name)
    if '=' in text:
        left, right = text.split('=', 1)
        return f'{left} == {right}'
    return text


def make_term_text(feature_name: Any, term_label: str) -> str:
    readable = humanize_feature_name(feature_name)
    if '=' in str(feature_name):
        if term_label == 'high':
            return f'{readable} is active'
        if term_label == 'low':
            return f'{readable} is inactive'
        return f'{readable} is partially active'
    return f'{readable} is {term_label}'


def focus_feature_indices_from_sample(sample_feature_df: pd.DataFrame, top_terms: int = 8) -> list[int]:
    row = sample_feature_df.iloc[0].to_numpy(dtype=float)
    non_zero = [idx for idx, value in enumerate(row) if abs(float(value)) > 1e-12]
    ordered = []
    seen = set()
    for idx in non_zero:
        if idx not in seen:
            ordered.append(idx)
            seen.add(idx)
    if len(ordered) < int(top_terms):
        for idx in np.argsort(np.abs(row))[::-1].tolist():
            if idx not in seen:
                ordered.append(idx)
                seen.add(idx)
            if len(ordered) >= int(top_terms):
                break
    return ordered[: max(1, int(top_terms))]


def tsk_rule_indices(model: TSKANFIS) -> np.ndarray:
    if hasattr(model, 'rule_mf_indices') and model.rule_mf_indices is not None:
        return model.rule_mf_indices.detach().cpu().numpy().astype(int)
    selector_logits = model.rule_mf_selector_logits.detach().cpu().numpy()
    return selector_logits.argmax(axis=-1).astype(int)


def build_tsk_rule_texts(model: TSKANFIS, feature_names: list[str], focus_feature_idx: list[int]) -> list[str]:
    centers = model.centers.detach().cpu().numpy()
    rule_idx = tsk_rule_indices(model)
    texts = []
    for r in range(int(model.n_rules)):
        terms = []
        for d in focus_feature_idx:
            feat = feature_names[d]
            term_label, _ = infer_term_label(centers[d], int(rule_idx[r, d]))
            terms.append(make_term_text(feat, term_label))
        texts.append(' AND '.join(terms))
    return texts


@torch.no_grad()
def predict_binary_logits(model, x_tensor: torch.Tensor, batch_size: int = 512) -> np.ndarray:
    model.eval()
    outs = []
    n = int(x_tensor.shape[0])
    for start in range(0, n, batch_size):
        logits = model(x_tensor[start:start + batch_size])
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        elif logits.dim() != 1:
            raise ValueError(f'Only binary single-logit models are supported here, got {tuple(logits.shape)}')
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


@torch.no_grad()
def analyze_tsk_sample(
    model: TSKANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    sample_feature_df: pd.DataFrame | None = None,
    top_k: int = 3,
    top_terms: int = 8,
    label: str = 'TSK-ANFIS',
):
    model.eval()
    logits, acts = model(x_tensor_2d, return_activations=True)
    if logits.dim() == 2 and logits.size(1) == 1:
        final_logit = float(logits.detach().cpu().reshape(-1)[0])
    elif logits.dim() == 1:
        final_logit = float(logits.detach().cpu().reshape(-1)[0])
    else:
        raise ValueError(f'Only binary single-logit models are supported here, got {tuple(logits.shape)}')

    final_prob = float(sigmoid_np(final_logit))
    pred_class = int(final_prob >= 0.5)
    x_arr = x_tensor_2d.detach().cpu().numpy()[0]

    if sample_feature_df is not None:
        focus_feature_idx = focus_feature_indices_from_sample(sample_feature_df, top_terms=top_terms)
    else:
        focus_feature_idx = list(range(min(len(feature_names), int(top_terms))))

    rule_weights = acts['rule_weights'].detach().cpu().numpy()[0]
    rule_firing = acts.get('rule_firing')
    if rule_firing is not None:
        rule_firing = rule_firing.detach().cpu().numpy()[0]

    consequents = model.consequents.detach().cpu().numpy()
    rule_texts = build_tsk_rule_texts(model, feature_names, focus_feature_idx)

    rows = []
    for r in range(int(model.n_rules)):
        rule_logit = float(np.dot(consequents[r, 0, :-1], x_arr) + consequents[r, 0, -1])
        rule_prob = float(sigmoid_np(rule_logit))
        rule_weight = float(rule_weights[r])
        firing = float(rule_firing[r]) if rule_firing is not None else np.nan
        rows.append(
            {
                'rule_idx': int(r),
                'rule_id': f'R{r + 1}',
                'rule_weight': rule_weight,
                'rule_firing': firing,
                'rule_logit': rule_logit,
                'rule_prob_class_1': rule_prob,
                'rule_contribution_logit': float(rule_weight * rule_logit),
                'antecedent_text': rule_texts[r],
                'if_then_text': f'IF {rule_texts[r]} THEN class_1 probability = {rule_prob:.4f} (rule_logit={rule_logit:.4f})',
            }
        )

    all_rules_df = pd.DataFrame(rows).sort_values(['rule_weight', 'rule_contribution_logit'], ascending=[False, False]).reset_index(drop=True)
    return {
        'model_label': label,
        'final_logit': final_logit,
        'final_prob': final_prob,
        'pred_class': pred_class,
        'focus_feature_names': [feature_names[idx] for idx in focus_feature_idx],
        'top_rules_df': all_rules_df.head(int(top_k)).copy(),
        'all_rules_df': all_rules_df,
    }


@torch.no_grad()
def analyze_ph_sample(
    model: ParallelHierarchicalTSKANFIS,
    x_tensor_2d: torch.Tensor,
    feature_names: list[str],
    sample_feature_df: pd.DataFrame,
    top_k: int = 3,
    top_terms: int = 8,
    label: str = 'H-ANFIS',
):
    model.eval()
    y, acts = model(x_tensor_2d, return_activations=True)
    final_logit = float(y.detach().cpu().reshape(-1)[0])
    final_prob = float(sigmoid_np(final_logit))
    pred_class = int(final_prob >= 0.5)

    xa, xb = model.split_inputs(x_tensor_2d)
    feat_a = [feature_names[i] for i in model.group_a_idx]
    feat_b = [feature_names[i] for i in model.group_b_idx]
    sample_a = sample_feature_df.iloc[:, model.group_a_idx].copy()
    sample_b = sample_feature_df.iloc[:, model.group_b_idx].copy()

    branch_a_analysis = analyze_tsk_sample(model.branch_a, xa, feat_a, sample_feature_df=sample_a, top_k=top_k, top_terms=top_terms, label='Branch A')
    branch_b_analysis = analyze_tsk_sample(model.branch_b, xb, feat_b, sample_feature_df=sample_b, top_k=top_k, top_terms=top_terms, label='Branch B')

    out = {
        'model_label': label,
        'fusion': model.fusion,
        'final_logit': final_logit,
        'final_prob': final_prob,
        'pred_class': pred_class,
        'branch_a_analysis': branch_a_analysis,
        'branch_b_analysis': branch_b_analysis,
    }

    if model.fusion == 'avg':
        fusion_weight = float(torch.sigmoid(model.fusion_logits).detach().cpu().reshape(-1)[0])
        out['fusion_text'] = f'final_logit = {fusion_weight:.4f} * branch_a_logit + {1.0 - fusion_weight:.4f} * branch_b_logit'
        out['fusion_weight'] = fusion_weight
    else:
        top_input = acts['top_input']
        top_feature_names = [f'branch_output_{idx}' for idx in range(top_input.shape[1])]
        top_analysis = analyze_tsk_sample(model.top_anfis, top_input, top_feature_names, sample_feature_df=None, top_k=top_k, top_terms=min(top_terms, len(top_feature_names)), label='Top fusion ANFIS')
        out['fusion_text'] = 'stacked top-level ANFIS combines the two branch outputs.'
        out['top_analysis'] = top_analysis

    return out


def gh_branch_masks(model: GH_ANFIS) -> dict[str, np.ndarray]:
    base_soft = torch.sigmoid(model.base_mask_logits).detach().cpu().numpy()
    base_hard = (base_soft >= float(model.base_mask_threshold)).astype(float)
    residual_soft = torch.sigmoid(model.residual_mask_logits).detach().cpu().numpy()
    residual_hard = (residual_soft >= float(model.residual_mask_threshold)).astype(float)
    if bool(model.residual_use_complement):
        residual_effective_hard = residual_hard * (1.0 - base_hard)
    else:
        residual_effective_hard = residual_hard.copy()
    return {
        'base_soft': base_soft,
        'base_hard': base_hard,
        'residual_soft': residual_soft,
        'residual_hard': residual_hard,
        'residual_effective_hard': residual_effective_hard,
    }


def gh_module_pack(model: GH_ANFIS, module: str) -> dict[str, Any]:
    masks = gh_branch_masks(model)
    if module == 'base':
        return {
            'module': 'base',
            'module_label': 'Primary',
            'display_prefix': 'P',
            'internal_prefix': 'B',
            'centers': model.s_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.s_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.base_consequents,
            'gate_hard': masks['base_hard'],
            'n_rules': int(model.base_rules),
        }
    if module == 'residual':
        return {
            'module': 'residual',
            'module_label': 'Complementary',
            'display_prefix': 'C',
            'internal_prefix': 'R',
            'centers': model.p_mf_centers.detach().cpu().numpy(),
            'selector_logits': model.p_rule_mf_selector_logits.detach().cpu().numpy(),
            'consequents': model.residual_consequents,
            'gate_hard': masks['residual_effective_hard'],
            'n_rules': int(model.residual_rules),
        }
    raise ValueError("module must be 'base' or 'residual'")


@torch.no_grad()
def gh_predict_binary_logits(model: GH_ANFIS, x_tensor: torch.Tensor, phase: str, mode: str, batch_size: int = 512) -> np.ndarray:
    model.eval()
    model.set_phase(phase)
    model.set_mode(mode)
    outs = []
    n = int(x_tensor.shape[0])
    for start in range(0, n, batch_size):
        logits = model(x_tensor[start:start + batch_size])
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        elif logits.dim() != 1:
            raise ValueError(f'Only binary single-logit GH models are supported here, got {tuple(logits.shape)}')
        outs.append(logits.detach().cpu().numpy())
    return np.concatenate(outs, axis=0)


def gh_module_rule_details(model: GH_ANFIS, x_tensor_2d: torch.Tensor, feature_names: list[str], module: str, acts: dict[str, Any], top_terms: int = 8) -> pd.DataFrame:
    mp = gh_module_pack(model, module)
    selector_logits = np.asarray(mp['selector_logits'], dtype=np.float64)
    selector_logits = selector_logits - selector_logits.max(axis=-1, keepdims=True)
    selector_probs = np.exp(selector_logits)
    selector_probs = selector_probs / selector_probs.sum(axis=-1, keepdims=True)

    centers = np.asarray(mp['centers'], dtype=np.float64)
    x_arr = x_tensor_2d.detach().cpu().numpy()[0]
    if module == 'base':
        gate_vec = acts['gate_base'][0].detach().cpu().numpy()
        rule_weights = acts['base_rule_weights'][0].detach().cpu().numpy()
    else:
        gate_vec = acts['gate_residual'][0].detach().cpu().numpy()
        rule_weights = acts['residual_rule_weights'][0].detach().cpu().numpy()

    rows = []
    for r in range(mp['n_rules']):
        best_mf = selector_probs[r].argmax(axis=1)
        best_prob = selector_probs[r].max(axis=1)
        active_idx = np.where(gate_vec > 1e-8)[0].tolist()
        if active_idx:
            active_idx = sorted(active_idx, key=lambda d: float(best_prob[d]), reverse=True)
        else:
            active_idx = np.argsort(best_prob)[::-1].tolist()
        active_idx = active_idx[: max(1, int(top_terms))]

        terms = []
        for d in active_idx:
            term_label, _ = infer_term_label(centers[d], int(best_mf[d]))
            terms.append(make_term_text(feature_names[d], term_label))
        antecedent_text = ' AND '.join(terms) if terms else '(no active terms)'

        layer = mp['consequents'][r]
        weight = layer.weight.detach().cpu().numpy().reshape(-1)
        bias = float(layer.bias.detach().cpu().numpy().reshape(-1)[0]) if layer.bias is not None else 0.0
        feat_weight = weight[: len(feature_names)]
        bias_weight = float(weight[len(feature_names)]) if len(weight) > len(feature_names) else 0.0
        rule_logit = float(np.sum(x_arr * feat_weight * gate_vec) + bias_weight + bias)
        rule_prob = float(sigmoid_np(rule_logit))
        rule_weight = float(rule_weights[r])

        rows.append(
            {
                'module': module,
                'module_label': mp['module_label'],
                'rule_idx': int(r),
                'display_rule_id': f"{mp['display_prefix']}{r + 1}",
                'internal_rule_id': f"{mp['internal_prefix']}{r + 1}",
                'rule_weight': rule_weight,
                'rule_logit': rule_logit,
                'rule_prob_class_1': rule_prob,
                'rule_contribution_logit': float(rule_weight * rule_logit),
                'antecedent_text': antecedent_text,
                'if_then_text': f'IF {antecedent_text} THEN class_1 probability = {rule_prob:.4f} (rule_logit={rule_logit:.4f})',
            }
        )

    return pd.DataFrame(rows).sort_values(['rule_weight', 'rule_contribution_logit'], ascending=[False, False]).reset_index(drop=True)


@torch.no_grad()
def analyze_gh_sample(model: GH_ANFIS, x_tensor_2d: torch.Tensor, feature_names: list[str], top_k_rules: int = 3, top_terms: int = 8, label: str = 'GRS-ANFIS'):
    base_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='base', mode='base_only', batch_size=1)[0])
    residual_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='residual_complement', mode='residual_only', batch_size=1)[0])
    full_logit = float(gh_predict_binary_logits(model, x_tensor_2d, phase='residual_complement', mode='full', batch_size=1)[0])

    base_prob = float(sigmoid_np(base_logit))
    residual_prob = float(sigmoid_np(residual_logit))
    full_prob = float(sigmoid_np(full_logit))

    model.eval()
    model.set_phase('residual_complement')
    model.set_mode('full')
    _, acts = model(x_tensor_2d, return_activations=True, use_soft_eval=False)

    base_rules_df = gh_module_rule_details(model, x_tensor_2d, feature_names, module='base', acts=acts, top_terms=top_terms)
    residual_rules_df = gh_module_rule_details(model, x_tensor_2d, feature_names, module='residual', acts=acts, top_terms=top_terms)

    return {
        'model_label': label,
        'base_logit': base_logit,
        'base_prob': base_prob,
        'base_pred': int(base_prob >= 0.5),
        'residual_logit': residual_logit,
        'residual_prob': residual_prob,
        'residual_pred': int(residual_prob >= 0.5),
        'full_logit': full_logit,
        'full_prob': full_prob,
        'full_pred': int(full_prob >= 0.5),
        'delta_full_minus_base': float(full_prob - base_prob),
        'base_rules_df': base_rules_df.head(int(top_k_rules)).copy(),
        'residual_rules_df': residual_rules_df.head(int(top_k_rules)).copy(),
        'all_base_rules_df': base_rules_df,
        'all_residual_rules_df': residual_rules_df,
    }


def gh_case_table_for_fold(fold_idx: int, artifact_dir: Path):
    train_idx, val_idx = get_bcwd_fold_indices(Y_ENC, fold_idx)
    model, payload = load_cv_artifact('GH-ANFIS', fold_idx, artifact_dir=artifact_dir)
    X_val_processed = PROCESSED_X_DF.iloc[val_idx].copy()
    _, X_val_scaled = preprocess_with_payload(X_val_processed, payload)
    x_tensor = torch.tensor(X_val_scaled, dtype=torch.float32, device=DEVICE)

    base_logits = gh_predict_binary_logits(model, x_tensor, phase='base', mode='base_only')
    residual_logits = gh_predict_binary_logits(model, x_tensor, phase='residual_complement', mode='residual_only')
    full_logits = gh_predict_binary_logits(model, x_tensor, phase='residual_complement', mode='full')

    df = pd.DataFrame(
        {
            'fold': int(fold_idx),
            'orig_index': val_idx,
            'y_raw': Y_RAW[val_idx],
            'y_encoded': Y_ENC[val_idx],
            'base_prob': sigmoid_np(base_logits),
            'residual_prob': sigmoid_np(residual_logits),
            'full_prob': sigmoid_np(full_logits),
        }
    )
    df['base_pred'] = (df['base_prob'] >= 0.5).astype(int)
    df['residual_pred'] = (df['residual_prob'] >= 0.5).astype(int)
    df['full_pred'] = (df['full_prob'] >= 0.5).astype(int)
    df['delta_full_minus_base'] = df['full_prob'] - df['base_prob']
    df['is_malignant_truth'] = df['y_encoded'] == MALIGNANT_ENC_LABEL
    df['base_benign_full_malignant'] = (df['base_pred'] == 0) & (df['full_pred'] == 1)
    return df.sort_values(['base_benign_full_malignant', 'delta_full_minus_base', 'full_prob'], ascending=[False, False, False]).reset_index(drop=True)


def find_gh_recovery_case_across_folds(search_folds: list[int]):
    fold_tables = []
    for fold_idx in search_folds:
        artifact_dir = resolve_artifact_dataset_dir(DATASET_NAME, fold_idx, MODEL_NAMES, preferred_dirname=ARTIFACT_DATASET_DIRNAME)
        fold_df = gh_case_table_for_fold(fold_idx, artifact_dir=artifact_dir)
        fold_tables.append(fold_df)

    all_cases_df = pd.concat(fold_tables, axis=0, ignore_index=True)
    target_df = all_cases_df[
        (all_cases_df['is_malignant_truth'])
        & (all_cases_df['base_pred'] == 0)
        & (all_cases_df['full_pred'] == 1)
    ].copy()

    if target_df.empty:
        raise RuntimeError('No BCWD case satisfies malignant truth + base benign + full malignant for the current checkpoints.')

    selected = target_df.sort_values(['delta_full_minus_base', 'full_prob', 'residual_prob'], ascending=[False, False, False]).iloc[0]
    return all_cases_df, target_df.reset_index(drop=True), int(selected['fold']), int(selected['orig_index'])


def describe_gh_recovery_case(analysis: dict[str, Any]) -> str:
    return (
        f"Primary(base)는 benign 쪽인 class_{analysis['base_pred']}로 시작합니다 "
        f"(base_prob={analysis['base_prob']:.4f}). "
        f"Complementary(residual)는 malignant 쪽인 class_{analysis['residual_pred']}를 강하게 지지하고 "
        f"(residual_prob={analysis['residual_prob']:.4f}), "
        f"최종 full은 class_{analysis['full_pred']}로 회복됩니다 "
        f"(full_prob={analysis['full_prob']:.4f}, delta={analysis['delta_full_minus_base']:.4f})."
    )


def build_model_input_view(sample_processed_df: pd.DataFrame) -> pd.DataFrame:
    row = sample_processed_df.iloc[0]
    active = row[row != 0].sort_values(ascending=False).reset_index()
    active.columns = ['feature', 'value']
    active['feature_readable'] = active['feature'].map(humanize_feature_name)
    return active[['feature_readable', 'feature', 'value']]


In [ ]:
all_gh_cases_df, recovery_matches_df, selected_fold_idx, selected_case_index = find_gh_recovery_case_across_folds(SEARCH_FOLDS)
case_selection_reason = 'ground truth malignant + GH base benign + GH full malignant'

if FOLD_IDX_OVERRIDE is not None:
    selected_fold_idx = int(FOLD_IDX_OVERRIDE)
if CASE_INDEX_OVERRIDE is not None:
    selected_case_index = int(CASE_INDEX_OVERRIDE)
    case_selection_reason = 'manual override'

FOLD_IDX = int(selected_fold_idx)
TRAIN_IDX, VAL_IDX = get_bcwd_fold_indices(Y_ENC, FOLD_IDX)
ARTIFACT_DATASET_DIR = resolve_artifact_dataset_dir(DATASET_NAME, FOLD_IDX, MODEL_NAMES, preferred_dirname=ARTIFACT_DATASET_DIRNAME)

selected_case_raw = RAW_X_DF.iloc[[selected_case_index]].copy()
selected_case_processed = PROCESSED_X_DF.iloc[[selected_case_index]].copy()
selected_case_input_view = build_model_input_view(selected_case_processed)
selected_case_target_raw = int(Y_RAW[selected_case_index])
selected_case_target_encoded = int(Y_ENC[selected_case_index])
selected_case_is_val = bool(int(selected_case_index) in set(VAL_IDX.tolist()))

selected_row = recovery_matches_df[(recovery_matches_df['fold'] == FOLD_IDX) & (recovery_matches_df['orig_index'] == selected_case_index)]
if selected_row.empty and case_selection_reason != 'manual override':
    raise RuntimeError('Selected case was not found in recovery matches.')

print('Selected fold:', FOLD_IDX)
print('Selected case index:', selected_case_index)
print('Selection reason:', case_selection_reason)
print('Raw target:', selected_case_target_raw)
print('Encoded target:', selected_case_target_encoded)
print('Is in validation fold:', selected_case_is_val)
print('Malignant raw label:', MALIGNANT_RAW_LABEL)
print('Benign raw label:', BENIGN_RAW_LABEL)
print('Model input feature count:', len(selected_case_input_view))

print('All GH cases that satisfy the requested condition:')
display(recovery_matches_df)

print('Selected patient raw feature values (9 original variables):')
display(selected_case_raw.T.rename(columns={selected_case_index: 'value'}))

print('Selected patient model input feature values (before scaling):')
display(selected_case_input_view)


In [ ]:
analysis_by_model = {}
summary_rows = []

for model_name in MODEL_NAMES:
    model, payload = load_cv_artifact(model_name, FOLD_IDX, artifact_dir=ARTIFACT_DATASET_DIR)
    sample_feature_df, sample_scaled = preprocess_with_payload(selected_case_processed, payload)
    sample_tensor = torch.tensor(sample_scaled, dtype=torch.float32, device=DEVICE)

    if model_name == 'GH-ANFIS':
        analysis = analyze_gh_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            top_k_rules=TOP_RULES_PER_MODEL,
            top_terms=TOP_TERMS_PER_RULE,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['full_pred']),
                'class_1_probability': float(analysis['full_prob']),
                'base_probability': float(analysis['base_prob']),
                'residual_probability': float(analysis['residual_prob']),
                'delta_full_minus_base': float(analysis['delta_full_minus_base']),
                'top_rule_summary': (
                    f"base={analysis['base_rules_df'].iloc[0]['display_rule_id'] if not analysis['base_rules_df'].empty else 'NA'}; "
                    f"residual={analysis['residual_rules_df'].iloc[0]['display_rule_id'] if not analysis['residual_rules_df'].empty else 'NA'}"
                ),
            }
        )
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        analysis = analyze_ph_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            sample_feature_df=sample_feature_df,
            top_k=TOP_RULES_PER_MODEL,
            top_terms=TOP_TERMS_PER_RULE,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        top_rule_summary = 'branch_a=' + analysis['branch_a_analysis']['top_rules_df'].iloc[0]['rule_id']
        top_rule_summary += '; branch_b=' + analysis['branch_b_analysis']['top_rules_df'].iloc[0]['rule_id']
        if 'top_analysis' in analysis:
            top_rule_summary += '; top=' + analysis['top_analysis']['top_rules_df'].iloc[0]['rule_id']
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['pred_class']),
                'class_1_probability': float(analysis['final_prob']),
                'base_probability': np.nan,
                'residual_probability': np.nan,
                'delta_full_minus_base': np.nan,
                'top_rule_summary': top_rule_summary,
            }
        )
    else:
        analysis = analyze_tsk_sample(
            model,
            sample_tensor,
            list(sample_feature_df.columns),
            sample_feature_df=sample_feature_df,
            top_k=TOP_RULES_PER_MODEL,
            top_terms=TOP_TERMS_PER_RULE,
            label=MODEL_DISPLAY_NAMES[model_name],
        )
        summary_rows.append(
            {
                'artifact_name': model_name,
                'model': MODEL_DISPLAY_NAMES[model_name],
                'n_features_used': int(sample_feature_df.shape[1]),
                'predicted_class': int(analysis['pred_class']),
                'class_1_probability': float(analysis['final_prob']),
                'base_probability': np.nan,
                'residual_probability': np.nan,
                'delta_full_minus_base': np.nan,
                'top_rule_summary': analysis['top_rules_df'].iloc[0]['rule_id'],
            }
        )

    analysis['payload'] = payload
    analysis['sample_feature_df'] = sample_feature_df
    analysis_by_model[model_name] = analysis

comparison_df = pd.DataFrame(summary_rows)
comparison_df['true_encoded_class'] = selected_case_target_encoded
comparison_df['true_raw_target'] = selected_case_target_raw
comparison_df['correct'] = comparison_df['predicted_class'] == selected_case_target_encoded
comparison_df['sort_key'] = comparison_df['artifact_name'].map({name: idx for idx, name in enumerate(MODEL_NAMES)})
comparison_df = comparison_df.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

print('Model comparison summary for the selected recovery case:')
display(comparison_df)


In [ ]:
gh_analysis = analysis_by_model['GH-ANFIS']
gh_narrative = describe_gh_recovery_case(gh_analysis)

print('GRS-focused recovery interpretation:')
print(gh_narrative)

for model_name in MODEL_NAMES:
    analysis = analysis_by_model[model_name]
    display_name = MODEL_DISPLAY_NAMES[model_name]

    print('=' * 100)
    print(f'[{display_name}] artifact={model_name}')

    if model_name == 'GH-ANFIS':
        print(f"base_prob={analysis['base_prob']:.4f}, residual_prob={analysis['residual_prob']:.4f}, full_prob={analysis['full_prob']:.4f}")
        print(f"base_pred=class_{analysis['base_pred']}, full_pred=class_{analysis['full_pred']}")
        print('[Top Primary/base rules]')
        display(analysis['base_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        print('[Top Complementary/residual rules]')
        display(analysis['residual_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        print(f"final_prob={analysis['final_prob']:.4f}, pred=class_{analysis['pred_class']}")
        print('fusion:', analysis['fusion_text'])
        print('[Branch A top rules]')
        display(analysis['branch_a_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        print('[Branch B top rules]')
        display(analysis['branch_b_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        if 'top_analysis' in analysis:
            print('[Top fusion ANFIS rules]')
            display(analysis['top_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
    else:
        print(f"final_prob={analysis['final_prob']:.4f}, pred=class_{analysis['pred_class']}")
        display(analysis['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])


In [ ]:
def df_to_records(df: pd.DataFrame) -> list[dict[str, Any]]:
    if df is None or df.empty:
        return []
    return json.loads(df.to_json(orient='records', force_ascii=False))


stem = f'bcwd_fold_{int(FOLD_IDX):02d}_case_{int(selected_case_index):03d}_gh_recovery'
comparison_csv = EXPORT_DIR / f'{stem}_comparison.csv'
all_gh_cases_csv = EXPORT_DIR / f'{stem}_all_gh_cases.csv'
recovery_matches_csv = EXPORT_DIR / f'{stem}_recovery_matches.csv'
selected_case_raw_csv = EXPORT_DIR / f'{stem}_raw_features.csv'
selected_case_input_view_csv = EXPORT_DIR / f'{stem}_model_input_features.csv'
top_rules_json = EXPORT_DIR / f'{stem}_top_rules.json'
summary_json = EXPORT_DIR / f'{stem}_summary.json'

comparison_df.to_csv(comparison_csv, index=False)
all_gh_cases_df.to_csv(all_gh_cases_csv, index=False)
recovery_matches_df.to_csv(recovery_matches_csv, index=False)
selected_case_raw.T.rename(columns={selected_case_index: 'value'}).to_csv(selected_case_raw_csv)
selected_case_input_view.to_csv(selected_case_input_view_csv, index=False)

rules_payload = {}
for model_name in MODEL_NAMES:
    analysis = analysis_by_model[model_name]
    display_name = MODEL_DISPLAY_NAMES[model_name]
    if model_name == 'GH-ANFIS':
        rules_payload[display_name] = {
            'base_rules': df_to_records(analysis['base_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
            'residual_rules': df_to_records(analysis['residual_rules_df'][['display_rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }
    elif model_name in ('PH-ANFIS(Avg)', 'PH-ANFIS(Stacked)'):
        payload = {
            'branch_a_rules': df_to_records(analysis['branch_a_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
            'branch_b_rules': df_to_records(analysis['branch_b_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }
        if 'top_analysis' in analysis:
            payload['top_fusion_rules'] = df_to_records(analysis['top_analysis']['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']])
        rules_payload[display_name] = payload
    else:
        rules_payload[display_name] = {
            'rules': df_to_records(analysis['top_rules_df'][['rule_id', 'rule_weight', 'rule_prob_class_1', 'if_then_text']]),
        }

top_rules_json.write_text(json.dumps(rules_payload, ensure_ascii=False, indent=2), encoding='utf-8')

summary_payload = {
    'dataset_name': DATASET_NAME,
    'artifact_dataset_dir': str(ARTIFACT_DATASET_DIR),
    'fold_idx': int(FOLD_IDX),
    'selected_case_index': int(selected_case_index),
    'selection_reason': case_selection_reason,
    'selected_case_target_raw': int(selected_case_target_raw),
    'selected_case_target_encoded': int(selected_case_target_encoded),
    'is_validation_case': bool(selected_case_is_val),
    'label_mapping_raw_to_encoded': {str(k): int(v) for k, v in RAW_TO_ENC.items()},
    'gh_narrative': gh_narrative,
    'models': {
        row['artifact_name']: {
            'probability_class_1': float(row['class_1_probability']),
            'predicted_class': int(row['predicted_class']),
            'correct': bool(row['correct']),
        }
        for _, row in comparison_df.iterrows()
    },
}
summary_json.write_text(json.dumps(summary_payload, ensure_ascii=False, indent=2), encoding='utf-8')

print('Exported:')
print('  comparison_csv =', comparison_csv.resolve())
print('  all_gh_cases_csv =', all_gh_cases_csv.resolve())
print('  recovery_matches_csv =', recovery_matches_csv.resolve())
print('  selected_case_raw_csv =', selected_case_raw_csv.resolve())
print('  selected_case_input_view_csv =', selected_case_input_view_csv.resolve())
print('  top_rules_json =', top_rules_json.resolve())
print('  summary_json =', summary_json.resolve())
